# 08 — ME-STGCN-TA: kiểm chứng khoảng thời gian khi tính vận tốc

**Câu hỏi:** thay khoảng thời gian chung bằng khoảng thời gian danh định của từng cặp mốc có làm giảm bỏ sót té ngã mà không tăng báo nhầm quá mức không?

Notebook độc lập: không cần chạy 04–07 trước, không sửa notebook 07 hay model web 06. Hai nhánh đều dùng ME-STGCN-TA một luồng, 8 kênh, 33 khớp, 32 mốc, bốn block 64/64/128/128 và trọng số Fall ×2,25. Chỉ thay mẫu số của hai kênh vận tốc; tọa độ tương đối, visibility, góc thân và dịch chuyển hông giữ nguyên. Tăng cường dữ liệu biến đổi cả mốc thời gian khi crop.

**T0_legacy_dt:** khoảng thời gian = thời lượng video / 31, tái lập đầu vào A3_cost_225 của thí nghiệm 07. **T1_interval_dt:** khoảng thời gian = chênh lệch chỉ số frame gốc / FPS. Cách mới xử lý cả làm tròn chỉ số frame và việc frame cuối ở vị trí frame_count − 1, không phải frame_count.

**Giới hạn:** thời gian danh định không phải timestamp trình chiếu thực của từng frame. Vận tốc là thay đổi tọa độ tương đối đã chuẩn hóa cơ thể mỗi giây danh định, không phải m/s. Không sửa góc thân, nhãn hay chính sách điền pose. Dữ liệu đã được xem để phân tích lỗi: đây là thí nghiệm phát triển, không phải xác nhận mù trên người mới. Không hứa 95% accuracy và không chọn seed đẹp nhất.

**Input:** bốn output của notebook 01. Metadata FPS và frame_count đã có. Đã giải mã lại đủ 160 video tại máy, xác nhận số frame khớp manifest, đối chiếu quy tắc lấy mốc trong notebook 01 và nhúng bằng chứng vào notebook này. Không cần video gốc, MediaPipe hoặc input thứ năm trên Kaggle. Nếu dữ liệu khác bản đã kiểm tra, notebook dừng thay vì âm thầm đoán mốc.

Chạy **Run All**, cấu hình mặc định **full**. Full gồm 24 outer run và 96 lượt fit mạng, tối đa 50 epoch/lượt; early stopping có thể dừng sớm. Smoke chỉ kiểm tra phần mềm, không dùng làm kết quả nghiên cứu.

## Cell 1 — Kiểm tra môi trường
Bật GPU trước khi chạy full. Không cài lại PyTorch, NumPy hoặc MediaPipe. Chỉ dùng GPU đầu tiên; có hai T4 không đồng nghĩa notebook tự dùng cả hai.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import importlib.util
from pathlib import Path
import json
import hashlib
import sys
import shutil

required = ["numpy", "pandas", "torch", "sklearn", "matplotlib"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Thiếu thư viện: {missing}. Dùng notebook Python của Kaggle.")
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Image, FileLink
torch.set_num_threads(2)
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "output/research08"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Cell 2 — Nạp mã độc lập và bằng chứng timing
Cell dài vì đóng gói đủ mã để không phụ thuộc notebook cũ. Không cần chỉnh cell này. Mã đối chứng 07 được giữ nguyên; mã 08 nằm riêng. Hash được kiểm tra trước khi import.

In [ ]:
SOURCE_FILES = {'research07/features.py': 'import hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\nCHANNELS = ["relative_x", "relative_y", "visibility", "joint_velocity_x", "joint_velocity_y",\n            "torso_angle", "hip_displacement_x", "hip_displacement_y"]\nPAIRS = [(1, 4), (2, 5), (3, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16),\n         (17, 18), (19, 20), (21, 22), (23, 24), (25, 26), (27, 28), (29, 30), (31, 32)]\nFEATURE_VERSION = "v2-mask-visibility-raw-pose-augmentation"\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef find_unique(root, names):\n    matches = sorted({path for name in names for path in Path(root).rglob(name)})\n    if not matches:\n        raise FileNotFoundError(f"Missing input: {names}; attach notebook 01 outputs")\n    hashes = {sha256(path) for path in matches}\n    if len(hashes) != 1:\n        raise ValueError(f"Conflicting copies; remove ambiguous inputs: {matches}")\n    return matches[0]\n\n\ndef load_inputs(root):\n    paths = {\n        "pose": find_unique(root, ["gmdcsa24_pose_32.npz", "gmdcsa24_pose_32.zip"]),\n        "features": find_unique(root, ["gmdcsa24_features_32.npz", "gmdcsa24_features_32.zip"]),\n        "manifest": find_unique(root, ["gmdcsa24_pose_manifest.csv", "pose_mnf.txt"]),\n        "folds": find_unique(root, ["gmdcsa24_cross_subject_folds.csv", "cross.txt"]),\n    }\n    with np.load(paths["pose"], allow_pickle=False) as archive:\n        data = {key: archive[key].copy() for key in archive.files}\n    with np.load(paths["features"], allow_pickle=False) as archive:\n        features = {key: archive[key].copy() for key in archive.files}\n    for key in ["video_ids", "subjects", "labels", "masks"]:\n        if not np.array_equal(data[key], features[key]):\n            raise ValueError(f"Pose/feature alignment failed: {key}")\n    manifest = pd.read_csv(paths["manifest"])\n    folds = pd.read_csv(paths["folds"])\n    for column, key in [("video_id", "video_ids"), ("subject", "subjects"), ("label_id", "labels")]:\n        if not np.array_equal(manifest[column].to_numpy(), data[key]):\n            raise ValueError(f"Manifest order mismatch: {column}")\n    count = len(data["labels"])\n    if count != 160 or data["poses"].shape != (count, 32, 33, 4):\n        raise ValueError("This preregistered experiment expects 160 poses of shape (32,33,4)")\n    if len(np.unique(data["video_ids"])) != count or set(np.unique(data["subjects"])) != {1, 2, 3, 4}:\n        raise ValueError("Duplicate IDs or unexpected subjects")\n    if set(np.unique(data["labels"])) != {0, 1} or data["masks"].shape != (count, 32):\n        raise ValueError("Invalid labels or masks")\n    if not np.isfinite(data["poses"]).all() or not np.isfinite(features["durations"]).all() or np.any(features["durations"] <= 0):\n        raise ValueError("Nonfinite pose or invalid duration")\n    if len(folds) != count * 4 or folds["fold"].nunique() != 4:\n        raise ValueError("Saved split file does not have four folds")\n    for _, group in folds.groupby("fold"):\n        indices = group["sample_index"].to_numpy(dtype=int)\n        if not np.array_equal(np.sort(indices), np.arange(count)):\n            raise ValueError("Invalid saved sample indices")\n        if not np.array_equal(group["video_id"].to_numpy(), data["video_ids"][indices]):\n            raise ValueError("Saved fold IDs do not match pose order")\n        if not np.array_equal(group["subject"].to_numpy(), data["subjects"][indices]) or not np.array_equal(group["label_id"].to_numpy(), data["labels"][indices]):\n            raise ValueError("Saved fold subjects or labels mismatch")\n        partitions = [set(group.loc[group["split"] == split, "subject"]) for split in ["train", "val", "test"]]\n        if any(partitions[left] & partitions[right] for left, right in [(0, 1), (0, 2), (1, 2)]):\n            raise ValueError("Subject overlap in saved folds")\n    data.update(durations=features["durations"].astype(np.float32), manifest=manifest,\n                provenance={key: {"path": str(path), "sha256": sha256(path)} for key, path in paths.items()})\n    return data\n\n\ndef make_features_v2(pose, detected, duration, mask_visibility=True):\n    pose = np.asarray(pose, dtype=np.float32)\n    detected = np.asarray(detected, dtype=bool)\n    if pose.shape != (32, 33, 4) or detected.shape != (32,) or not np.isfinite(pose).all():\n        raise ValueError("Invalid finite pose/mask shape")\n    if not np.isfinite(duration) or duration <= 0:\n        raise ValueError("Invalid duration")\n    hips = pose[:, [23, 24], :2].mean(axis=1)\n    shoulders = pose[:, [11, 12], :2].mean(axis=1)\n    length = np.linalg.norm(shoulders - hips, axis=-1)\n    usable = length[length > 1e-3]\n    scale = max(float(np.median(usable)) if len(usable) else 1.0, 1e-3)\n    relative = (pose[..., :2] - hips[:, None, :]) / scale\n    velocity = np.zeros_like(relative)\n    velocity[1:] = np.diff(relative, axis=0) / max(float(duration) / 31, 1e-6)\n    visibility = np.clip(pose[..., 3], 0, 1).copy()\n    if mask_visibility:\n        visibility[~detected] = 0\n    torso = shoulders - hips\n    angle = np.arctan2(torso[:, 0], -torso[:, 1]) / np.pi\n    displacement = hips - hips[:1]\n    local = np.concatenate([relative, visibility[..., None], velocity], axis=-1)\n    global_values = np.concatenate([angle[:, None], displacement], axis=-1)\n    output = np.concatenate([local, np.repeat(global_values[:, None], 33, axis=1)], axis=-1)\n    return output.transpose(2, 0, 1).astype(np.float32)\n\n\ndef augment_pose(pose, detected, duration, rng, crop=True, flip=True, noise=True, dropout=True):\n    pose, detected = pose.copy(), detected.copy()\n    if crop and rng.random() < 0.5:\n        length = int(rng.integers(26, 33))\n        start = int(rng.integers(0, 33 - length))\n        positions = np.linspace(start, start + length - 1, 32)\n        lower, upper = np.floor(positions).astype(int), np.ceil(positions).astype(int)\n        fraction = (positions - lower).astype(np.float32)[:, None, None]\n        pose = pose[lower] * (1 - fraction) + pose[upper] * fraction\n        detected = detected[lower] & detected[upper]\n        duration *= (length - 1) / 31\n    if flip and rng.random() < 0.5:\n        pose[..., 0] = 1 - pose[..., 0]\n        original = pose.copy()\n        for left, right in PAIRS:\n            pose[:, left], pose[:, right] = original[:, right], original[:, left]\n    if noise and rng.random() < 0.35:\n        torso_length = np.linalg.norm(pose[:, [11, 12], :2].mean(axis=1) - pose[:, [23, 24], :2].mean(axis=1), axis=-1)\n        amplitude = max(float(np.median(torso_length)), 1e-3) * 0.01\n        pose[..., :2] += rng.normal(0, amplitude, pose[..., :2].shape).astype(np.float32)\n    dropped = np.empty(0, dtype=int)\n    if dropout and rng.random() < 0.2:\n        dropped = rng.choice(33, int(rng.integers(1, 4)), replace=False)\n    return pose, detected, float(duration), dropped\n\n\ndef svm_features(pose, detected, duration):\n    sequence = make_features_v2(pose, detected, duration)\n    speed = np.linalg.norm(sequence[3:5], axis=0)\n    height = np.ptp(sequence[1], axis=1)\n    height_velocity = np.diff(height, prepend=height[0]) / (float(duration) / 31)\n    per_frame = np.stack([speed.mean(axis=1), np.quantile(speed, 0.9, axis=1), sequence[5, :, 0],\n                          sequence[6, :, 0], sequence[7, :, 0], height, height_velocity], axis=1)\n    return np.concatenate([per_frame.mean(axis=0), per_frame.std(axis=0), per_frame.min(axis=0),\n                           per_frame.max(axis=0), *np.quantile(per_frame, [0.1, 0.5, 0.9], axis=0)]).astype(np.float32)\n', 'research07/models.py': 'import random\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom research07.features import augment_pose, make_features_v2\n\n\nEDGES = [(0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),(9,10),\n         (11,12),(11,13),(13,15),(15,17),(15,19),(15,21),(17,19),\n         (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),\n         (11,23),(12,24),(23,24),(23,25),(24,26),(25,27),(26,28),\n         (27,29),(28,30),(29,31),(30,32),(27,31),(28,32)]\n\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef make_adjacency():\n    adjacency = np.eye(33, dtype=np.float32)\n    for source, target in EDGES:\n        adjacency[source, target] = adjacency[target, source] = 1\n    inverse = np.maximum(adjacency.sum(axis=1), 1e-8) ** -0.5\n    return torch.from_numpy(inverse[:, None] * adjacency * inverse[None])\n\n\nclass SpatialGraphConv(nn.Module):\n    def __init__(self, in_channels, out_channels, adjacency):\n        super().__init__()\n        self.projection = nn.Conv2d(in_channels, out_channels, 1)\n        self.register_buffer("adjacency", adjacency.clone())\n        self.edge_importance = nn.Parameter(torch.ones_like(adjacency))\n\n    def forward(self, values):\n        return torch.einsum("nctv,vw->nctw", self.projection(values), self.adjacency * self.edge_importance)\n\n\nclass STGCNBlock(nn.Module):\n    def __init__(self, in_channels, out_channels, adjacency, stride=1, dropout=0.3):\n        super().__init__()\n        self.graph_conv = SpatialGraphConv(in_channels, out_channels, adjacency)\n        self.temporal_conv = nn.Sequential(nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),\n            nn.Conv2d(out_channels, out_channels, (9, 1), stride=(stride, 1), padding=(4, 0)),\n            nn.BatchNorm2d(out_channels), nn.Dropout(dropout))\n        self.residual = nn.Identity() if in_channels == out_channels and stride == 1 else nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, 1, stride=(stride, 1)), nn.BatchNorm2d(out_channels))\n        self.activation = nn.ReLU(inplace=True)\n\n    def forward(self, values):\n        return self.activation(self.temporal_conv(self.graph_conv(values)) + self.residual(values))\n\n\nclass SkeletonModel(nn.Module):\n    def __init__(self, channels=8, attention=True, dropout=0.3):\n        super().__init__()\n        adjacency = make_adjacency()\n        self.data_bn = nn.BatchNorm1d(channels * 33)\n        self.blocks = nn.Sequential(STGCNBlock(channels, 64, adjacency, dropout=dropout),\n            STGCNBlock(64, 64, adjacency, dropout=dropout),\n            STGCNBlock(64, 128, adjacency, stride=2, dropout=dropout),\n            STGCNBlock(128, 128, adjacency, dropout=dropout))\n        self.temporal_attention = nn.Sequential(nn.Conv1d(128, 64, 1), nn.Tanh(), nn.Conv1d(64, 1, 1)) if attention else None\n        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(128, 2))\n\n    def forward(self, values):\n        batch, channels, frames, nodes = values.shape\n        values = values.permute(0, 3, 1, 2).contiguous().view(batch, nodes * channels, frames)\n        values = self.data_bn(values).view(batch, nodes, channels, frames).permute(0, 2, 3, 1)\n        temporal = self.blocks(values).mean(dim=3)\n        weights = torch.softmax(self.temporal_attention(temporal), dim=2) if self.temporal_attention is not None else torch.ones_like(temporal[:, :1]) / temporal.shape[2]\n        return self.classifier((temporal * weights).sum(dim=2)), weights.squeeze(1)\n\n\nclass PoseDataset(Dataset):\n    def __init__(self, data, indices, channels, augment, seed):\n        self.data, self.indices, self.channels, self.augment = data, indices, channels, augment\n        self.rng = np.random.default_rng(seed)\n\n    def __len__(self):\n        return len(self.indices)\n\n    def __getitem__(self, offset):\n        index = self.indices[offset]\n        pose, mask, duration = self.data["poses"][index], self.data["masks"][index], self.data["durations"][index]\n        dropped = []\n        if self.augment:\n            pose, mask, duration, dropped = augment_pose(pose, mask, duration, self.rng)\n        features = make_features_v2(pose, mask, duration)\n        features[:5, :, dropped] = 0\n        return torch.from_numpy(features[:self.channels]), torch.tensor(int(self.data["labels"][index]), dtype=torch.long)\n\n\ndef make_loader(data, indices, variant, config, augment, seed):\n    return DataLoader(PoseDataset(data, indices, variant["channels"], augment, seed),\n                      batch_size=config["batch_size"], shuffle=augment, num_workers=0,\n                      generator=torch.Generator().manual_seed(seed))\n', 'research07/training.py': 'import json\nimport platform\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport sklearn\nimport torch\nfrom sklearn.metrics import confusion_matrix\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.svm import SVC\nfrom torch import nn\n\nfrom research07.features import FEATURE_VERSION, svm_features\nfrom research07.models import SkeletonModel, make_loader, set_seed\n\n\nVARIANTS = {\n    "A0_pose_average": {"channels": 3, "attention": False, "fall_multiplier": 1.0},\n    "A1_motion_average": {"channels": 8, "attention": False, "fall_multiplier": 1.0},\n    "A2_pose_attention": {"channels": 3, "attention": True, "fall_multiplier": 1.0},\n    "A3_motion_attention": {"channels": 8, "attention": True, "fall_multiplier": 1.0},\n    "A3_cost_225": {"channels": 8, "attention": True, "fall_multiplier": 2.25},\n}\n\n\ndef default_config(mode="full"):\n    if mode not in {"full", "smoke"}:\n        raise ValueError("mode must be full or smoke")\n    return {"mode": mode, "feature_version": FEATURE_VERSION, "seeds": [42, 142, 242] if mode == "full" else [42],\n            "variants": list(VARIANTS), "max_epochs": 50 if mode == "full" else 2,\n            "patience": 10 if mode == "full" else 1, "batch_size": 16, "learning_rate": 0.001,\n            "weight_decay": 0.0001, "dropout": 0.3, "scheduler": None,\n            "target_sensitivity": 0.95, "minimum_specificity": 0.75,\n            "threshold_grid": np.linspace(0.05, 0.95, 37).tolist(), "allow_cpu_full": False,\n            "epoch_selection": "minimum unweighted inner-validation cross entropy",\n            "protocol": "4 outer subjects; 3 inner subjects; refit outer train; no final all-subject retrain"}\n\n\ndef metrics(labels, predictions):\n    true_negative, false_positive, false_negative, true_positive = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()\n    ratio = lambda top, bottom: float(top / bottom) if bottom else None\n    sensitivity = ratio(true_positive, true_positive + false_negative)\n    specificity = ratio(true_negative, true_negative + false_positive)\n    return {"accuracy": ratio(true_positive + true_negative, len(labels)),\n            "balanced_accuracy": (sensitivity + specificity) / 2 if sensitivity is not None and specificity is not None else None,\n            "sensitivity": sensitivity, "specificity": specificity,\n            "precision": ratio(true_positive, true_positive + false_positive),\n            "f1": ratio(2 * true_positive, 2 * true_positive + false_positive + false_negative),\n            "f2": ratio(5 * true_positive, 5 * true_positive + false_positive + 4 * false_negative),\n            "false_alarm_rate": ratio(false_positive, false_positive + true_negative),\n            "tp": int(true_positive), "tn": int(true_negative), "fp": int(false_positive), "fn": int(false_negative)}\n\n\ndef select_thresholds(labels, scores, config, grid=None, fixed=0.5):\n    candidates = []\n    for threshold in config["threshold_grid"] if grid is None else grid:\n        values = metrics(labels, scores >= threshold)\n        meets_recall = values["sensitivity"] >= config["target_sensitivity"]\n        feasible = meets_recall and values["specificity"] >= config["minimum_specificity"]\n        tier = 2 if feasible else 1 if meets_recall else 0\n        safety_rank = (tier, values["specificity"], values["f2"], -abs(threshold - fixed)) if tier else (0, values["sensitivity"], values["specificity"], -abs(threshold - fixed))\n        candidates.append({"threshold": float(threshold), "metrics": values, "feasible": feasible,\n                           "balanced_rank": (values["balanced_accuracy"], -abs(threshold - fixed)), "safety_rank": safety_rank})\n    selected = {"fixed": {"threshold": fixed, "metrics": metrics(labels, scores >= fixed), "feasible": None},\n                "balanced": max(candidates, key=lambda item: item["balanced_rank"]),\n                "sensitivity": max(candidates, key=lambda item: item["safety_rank"])}\n    return selected\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_suffix(".pending")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef predict(model, loader, device):\n    model.eval()\n    labels, probabilities, attention = [], [], []\n    with torch.no_grad():\n        for features, target in loader:\n            logits, weights = model(features.to(device))\n            labels.extend(target.tolist())\n            probabilities.extend(logits.softmax(dim=1)[:, 1].cpu().tolist())\n            attention.extend(weights.cpu().tolist())\n    return np.asarray(labels), np.asarray(probabilities), np.asarray(attention)\n\n\ndef fit_model(data, train_indices, validation_indices, variant, config, seed, device, fixed_epochs=None):\n    set_seed(seed)\n    model = SkeletonModel(variant["channels"], variant["attention"], config["dropout"]).to(device)\n    train_loader = make_loader(data, train_indices, variant, config, True, seed)\n    counts = np.bincount(data["labels"][train_indices], minlength=2)\n    if np.any(counts == 0):\n        raise ValueError("Training fold has a missing class")\n    class_weights = len(train_indices) / (2 * counts.astype(np.float32))\n    class_weights[1] *= variant["fall_multiplier"]\n    criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))\n    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])\n    validation_loader = make_loader(data, validation_indices, variant, config, False, seed) if validation_indices is not None else None\n    best_loss, best_state, best_epoch, stale = float("inf"), None, 0, 0\n    history = []\n    for epoch in range(1, (fixed_epochs or config["max_epochs"]) + 1):\n        model.train()\n        total_loss = 0\n        for features, target in train_loader:\n            features, target = features.to(device), target.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            logits, _ = model(features)\n            loss = criterion(logits, target)\n            loss.backward()\n            optimizer.step()\n            total_loss += loss.item() * len(target)\n        record = {"epoch": epoch, "train_weighted_loss": total_loss / len(train_indices)}\n        if validation_loader is not None:\n            labels, scores, _ = predict(model, validation_loader, device)\n            scores = np.clip(scores, 1e-7, 1 - 1e-7)\n            validation_loss = float(-(labels * np.log(scores) + (1 - labels) * np.log(1 - scores)).mean())\n            record["validation_unweighted_loss"] = validation_loss\n            if validation_loss < best_loss - 1e-6:\n                best_loss, best_epoch, stale = validation_loss, epoch, 0\n                best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}\n            else:\n                stale += 1\n        history.append(record)\n        if validation_loader is not None and stale >= config["patience"]:\n            break\n    if best_state is not None:\n        model.load_state_dict(best_state)\n    return model, best_epoch if validation_loader is not None else fixed_epochs, history\n\n\ndef run_neural(data, config, output, device):\n    subjects = sorted(np.unique(data["subjects"]).tolist())\n    for variant_name in config["variants"]:\n        variant = VARIANTS[variant_name]\n        for seed in config["seeds"]:\n            for outer in subjects:\n                name = f"{variant_name}_seed{seed}_subject{outer}"\n                target = output / f"{name}.json"\n                if target.exists():\n                    print("Resume completed:", name, flush=True)\n                    continue\n                train_indices = np.flatnonzero(data["subjects"] != outer)\n                test_indices = np.flatnonzero(data["subjects"] == outer)\n                inner_scores, inner_labels, inner_epochs, histories, splits = [], [], [], [], []\n                for inner in subjects:\n                    if inner == outer:\n                        continue\n                    fit_indices = np.flatnonzero((data["subjects"] != outer) & (data["subjects"] != inner))\n                    val_indices = np.flatnonzero(data["subjects"] == inner)\n                    assert not set(fit_indices) & set(val_indices) and not set(train_indices) & set(test_indices)\n                    run_seed = seed + outer * 1000 + inner * 10\n                    model, best_epoch, history = fit_model(data, fit_indices, val_indices, variant, config, run_seed, device)\n                    labels, scores, _ = predict(model, make_loader(data, val_indices, variant, config, False, run_seed), device)\n                    inner_scores.extend(scores.tolist())\n                    inner_labels.extend(labels.tolist())\n                    inner_epochs.append(best_epoch)\n                    histories.extend([{**row, "inner_subject": inner} for row in history])\n                    splits.append({"inner_subject": inner, "train_indices": fit_indices.tolist(), "val_indices": val_indices.tolist(), "best_epoch": best_epoch})\n                    del model\n                choices = select_thresholds(np.asarray(inner_labels), np.asarray(inner_scores), config)\n                epochs = max(1, int(np.median(inner_epochs)))\n                model, _, refit_history = fit_model(data, train_indices, None, variant, config, seed + outer * 1000 + 999, device, epochs)\n                labels, scores, attention = predict(model, make_loader(data, test_indices, variant, config, False, seed), device)\n                predictions = []\n                fold_metrics = []\n                for policy, choice in choices.items():\n                    threshold = choice["threshold"]\n                    predicted = (scores >= threshold).astype(int)\n                    fold_metrics.append({"variant": variant_name, "seed": seed, "subject": outer, "policy": policy,\n                                         "threshold": threshold, "epochs": epochs, "inner_target_met": choice["feasible"], **metrics(labels, predicted)})\n                    for index, label, score, prediction in zip(test_indices, labels, scores, predicted):\n                        predictions.append({"sample_index": int(index), "video_id": str(data["video_ids"][index]), "subject": outer,\n                            "label": int(label), "score": float(score), "prediction": int(prediction), "threshold": threshold,\n                            "variant": variant_name, "seed": seed, "policy": policy})\n                checkpoint = {"model_state_dict": {key: value.cpu() for key, value in model.state_dict().items()},\n                              "variant": variant, "feature_version": FEATURE_VERSION, "train_indices": train_indices.tolist(),\n                              "test_indices": test_indices.tolist(), "config": config, "epochs": epochs}\n                torch.save(checkpoint, output / f"{name}.pt")\n                np.savez_compressed(output / f"{name}_attention.npz", indices=test_indices, attention=attention)\n                pd.DataFrame(histories).to_csv(output / f"{name}_inner_history.csv", index=False)\n                pd.DataFrame(refit_history).to_csv(output / f"{name}_refit_history.csv", index=False)\n                write_json(target, {"complete": True, "inner_splits": splits, "inner_labels": inner_labels, "inner_scores": inner_scores,\n                                    "choices": choices, "metrics": fold_metrics, "predictions": predictions,\n                                    "parameters": sum(parameter.numel() for parameter in model.parameters())})\n                print("Completed", name, "epochs", epochs, flush=True)\n                del model\n\n\ndef run_svm(data, config, output):\n    values = np.stack([svm_features(pose, mask, duration) for pose, mask, duration in zip(data["poses"], data["masks"], data["durations"])])\n    subjects = sorted(np.unique(data["subjects"]).tolist())\n    for outer in subjects:\n        target = output / f"SVM_v2_seed0_subject{outer}.json"\n        if target.exists():\n            continue\n        candidates = []\n        grid = [(1, "scale")] if config["mode"] == "smoke" else [(regularization, gamma) for regularization in [0.1, 1, 10] for gamma in ["scale", 0.01]]\n        for regularization, gamma in grid:\n            labels_all, scores_all = [], []\n            for inner in subjects:\n                if inner == outer:\n                    continue\n                train = np.flatnonzero((data["subjects"] != outer) & (data["subjects"] != inner))\n                validation = np.flatnonzero(data["subjects"] == inner)\n                estimator = make_pipeline(StandardScaler(), SVC(C=regularization, gamma=gamma, class_weight="balanced"))\n                estimator.fit(values[train], data["labels"][train])\n                scores_all.extend(estimator.decision_function(values[validation]).tolist())\n                labels_all.extend(data["labels"][validation].tolist())\n            score = metrics(np.asarray(labels_all), np.asarray(scores_all) >= 0)["balanced_accuracy"]\n            candidates.append({"C": regularization, "gamma": gamma, "rank": score, "labels": labels_all, "scores": scores_all})\n        best = max(candidates, key=lambda item: item["rank"])\n        threshold_grid = np.unique(np.r_[np.quantile(best["scores"], np.linspace(0, 1, 37)), 0])\n        choices = select_thresholds(np.asarray(best["labels"]), np.asarray(best["scores"]), config, threshold_grid, fixed=0)\n        train, test = np.flatnonzero(data["subjects"] != outer), np.flatnonzero(data["subjects"] == outer)\n        estimator = make_pipeline(StandardScaler(), SVC(C=best["C"], gamma=best["gamma"], class_weight="balanced"))\n        estimator.fit(values[train], data["labels"][train])\n        scores = estimator.decision_function(values[test])\n        predictions, fold_metrics = [], []\n        for policy, choice in choices.items():\n            predicted = scores >= choice["threshold"]\n            fold_metrics.append({"variant": "SVM_v2", "seed": 0, "subject": outer, "policy": policy,\n                                 "threshold": choice["threshold"], "inner_target_met": choice["feasible"], **metrics(data["labels"][test], predicted)})\n            for index, score, prediction in zip(test, scores, predicted):\n                predictions.append({"sample_index": int(index), "video_id": str(data["video_ids"][index]), "subject": outer,\n                    "label": int(data["labels"][index]), "score": float(score), "prediction": int(prediction), "threshold": choice["threshold"],\n                    "variant": "SVM_v2", "seed": 0, "policy": policy})\n        write_json(target, {"complete": True, "choices": choices, "search": candidates, "metrics": fold_metrics, "predictions": predictions,\n                            "note": "49 motion statistics; scaler fit on training subjects only; scores are margins, not probabilities"})\n        print("Completed SVM_v2 subject", outer, flush=True)\n\n\ndef summarize_run(config, output):\n    paths = sorted(output.glob("*_seed*_subject*.json"))\n    expected = len(config["variants"]) * len(config["seeds"]) * 4 + 4\n    if len(paths) != expected:\n        raise ValueError(f"Only {len(paths)}/{expected} complete outer runs; no final summary")\n    reports = [json.loads(path.read_text(encoding="utf-8")) for path in paths]\n    predictions = pd.DataFrame([row for report in reports for row in report["predictions"]])\n    folds = pd.DataFrame([row for report in reports for row in report["metrics"]])\n    pooled = []\n    for (variant, seed, policy), group in predictions.groupby(["variant", "seed", "policy"]):\n        if len(group) != 160 or group["sample_index"].nunique() != 160:\n            raise ValueError("OOF coverage failed; do not pool duplicates across seeds")\n        pooled.append({"variant": variant, "seed": int(seed), "policy": policy, **metrics(group["label"], group["prediction"])})\n    pooled = pd.DataFrame(pooled)\n    for name, frame in [("predictions", predictions), ("fold_metrics", folds), ("pooled_by_seed", pooled),\n                        ("errors", predictions[predictions["label"] != predictions["prediction"]])]:\n        frame.to_csv(output / f"{name}.csv", index=False)\n    columns = ["accuracy", "balanced_accuracy", "sensitivity", "specificity", "precision", "f1", "f2", "false_alarm_rate"]\n    pooled.groupby(["variant", "policy"])[columns].agg(["mean", "std"]).to_csv(output / "seed_mean_std.csv")\n    write_json(output / "completion.json", {"complete": True, "mode": config["mode"], "scientific_result": config["mode"] == "full",\n               "outer_runs": len(paths), "warning": "Smoke results are software checks only. Repeated seeds do not increase independent sample count. Prior outer results have already informed development."})\n    return pooled\n\n\ndef run_experiment(data, config, output, code_hash):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    if device == "cpu" and config["mode"] == "full" and not config["allow_cpu_full"]:\n        raise RuntimeError("Full experiment requires Kaggle GPU; choose smoke locally, or explicitly allow CPU full")\n    signature = {"config": config, "inputs": {key: value["sha256"] for key, value in data["provenance"].items()}, "code_hash": code_hash,\n                 "versions": {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__, "sklearn": sklearn.__version__}, "device": device}\n    lock = output / "locked_protocol.json"\n    if lock.exists() and json.loads(lock.read_text(encoding="utf-8")) != signature:\n        raise ValueError("Run configuration/code/data/environment changed; use a NEW output directory")\n    write_json(lock, signature)\n    write_json(output / "input_provenance.json", data["provenance"])\n    started = time.monotonic()\n    run_neural(data, config, output, device)\n    run_svm(data, config, output)\n    summary = summarize_run(config, output)\n    write_json(output / "timing.json", {"seconds_this_invocation": time.monotonic() - started, "device": device})\n    return summary\n', 'research08/features.py': 'import copy\nimport hashlib\nimport json\n\nimport numpy as np\nimport pandas as pd\n\nfrom research07.features import augment_pose, make_features_v2\n\n\nFEATURE_VERSION = "08-interval-aware-nominal-timing-v1"\nFEATURE_MODES = {"legacy", "interval"}\nUNCHANGED_CHANNELS = [0, 1, 2, 5, 6, 7]\n\n\ndef attach_nominal_timing(data, evidence):\n    for name in ["pose", "features"]:\n        if data["provenance"][name]["sha256"] != evidence["input_hashes"][name]:\n            raise ValueError(f"Input {name} differs from verified notebook 01 data; attach the original saved outputs")\n    rows = evidence["rows"]\n    if len(rows) != len(data["labels"]) or evidence["samples"] != len(rows):\n        raise ValueError("Timing evidence does not cover all samples")\n    indices, times, diagnostics = [], [], []\n    for index, row in enumerate(rows):\n        metadata = data["manifest"].iloc[index]\n        if row["sample_index"] != index or row["video_id"] != str(data["video_ids"][index]):\n            raise ValueError("Timing evidence sample alignment failed")\n        frames = int(metadata["frame_count"])\n        fps = float(metadata["fps"])\n        if not np.isfinite(fps) or fps <= 0 or frames < 32:\n            raise ValueError(f"Invalid frame count/FPS: {row[\'video_id\']}")\n        if frames != row["decoded_frame_count"] or frames != row["frame_count"]:\n            raise ValueError(f"Decoded frame evidence mismatch: {row[\'video_id\']}")\n        if not np.isclose(fps, row["fps"], rtol=1e-9, atol=1e-9):\n            raise ValueError(f"FPS evidence mismatch: {row[\'video_id\']}")\n        expected_duration = frames / fps\n        if not np.isclose(data["durations"][index], expected_duration, rtol=1e-6, atol=1e-6):\n            raise ValueError(f"Duration mismatch: {row[\'video_id\']}")\n        if not np.isclose(float(metadata["duration_seconds"]), expected_duration, rtol=1e-6, atol=1e-6):\n            raise ValueError(f"Manifest duration mismatch: {row[\'video_id\']}")\n        positions = np.asarray(row["frame_indices"], dtype=np.int64)\n        expected = np.linspace(0, frames - 1, 32).round().astype(np.int64)\n        if positions.shape != (32,) or not np.array_equal(positions, expected) or np.any(np.diff(positions) <= 0):\n            raise ValueError(f"Sampling-rule mismatch: {row[\'video_id\']}")\n        timestamps = positions.astype(np.float64) / fps\n        actual_dt = np.diff(timestamps)\n        old_dt = float(data["durations"][index]) / 31\n        indices.append(positions)\n        times.append(timestamps)\n        diagnostics.append({"sample_index": index, "video_id": row["video_id"], "fps": fps,\n                            "decoded_frame_count": frames, "legacy_dt_seconds": old_dt,\n                            "nominal_dt_min_seconds": float(actual_dt.min()),\n                            "nominal_dt_max_seconds": float(actual_dt.max()),\n                            "max_relative_dt_difference": float(np.max(np.abs(actual_dt / old_dt - 1))),\n                            "legacy_duration_seconds": float(data["durations"][index]),\n                            "sampled_span_seconds": float(timestamps[-1] - timestamps[0])})\n    data["frame_indices"] = np.stack(indices)\n    data["timestamps"] = np.stack(times)\n    serialized = json.dumps(evidence, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)\n    data["timing_evidence_hash"] = hashlib.sha256(serialized.encode("utf-8")).hexdigest()\n    data["timing_evidence"] = evidence\n    return pd.DataFrame(diagnostics)\n\n\ndef validate_timestamps(timestamps):\n    timestamps = np.asarray(timestamps, dtype=np.float64)\n    if timestamps.shape != (32,) or not np.isfinite(timestamps).all() or np.any(np.diff(timestamps) <= 0):\n        raise ValueError("Require 32 finite, strictly increasing nominal timestamps")\n    return timestamps\n\n\ndef timed_features(pose, mask, duration, timestamps, mode):\n    if mode not in FEATURE_MODES:\n        raise ValueError(f"Unknown timing mode: {mode}")\n    timestamps = validate_timestamps(timestamps)\n    features = make_features_v2(pose, mask, duration)\n    if mode == "interval":\n        features[3:5, 1:] = np.diff(features[:2], axis=1) / np.diff(timestamps)[None, :, None]\n    if not np.isfinite(features).all():\n        raise ValueError("Nonfinite feature output")\n    return features\n\n\ndef augment_timed_pose(pose, mask, duration, timestamps, rng, crop=True, flip=True, noise=True, dropout=True):\n    timestamps = validate_timestamps(timestamps).copy()\n    probe = copy.deepcopy(rng)\n    if crop and probe.random() < 0.5:\n        length = int(probe.integers(26, 33))\n        start = int(probe.integers(0, 33 - length))\n        positions = np.linspace(start, start + length - 1, 32)\n        timestamps = np.interp(positions, np.arange(32), timestamps)\n    pose, mask, duration, dropped = augment_pose(pose, mask, duration, rng, crop=crop,\n                                                flip=flip, noise=noise, dropout=dropout)\n    return pose, mask, duration, validate_timestamps(timestamps), dropped\n', 'research08/dataset.py': 'import numpy as np\nimport torch\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom research08.features import augment_timed_pose, timed_features\n\n\nclass TimedPoseDataset(Dataset):\n    def __init__(self, data, indices, mode, augment, seed):\n        self.data = data\n        self.indices = np.asarray(indices, dtype=np.int64)\n        self.mode = mode\n        self.augment = augment\n        self.rng = np.random.default_rng(seed)\n\n    def __len__(self):\n        return len(self.indices)\n\n    def __getitem__(self, offset):\n        index = self.indices[offset]\n        pose = self.data["poses"][index]\n        mask = self.data["masks"][index]\n        duration = self.data["durations"][index]\n        timestamps = self.data["timestamps"][index]\n        dropped = np.empty(0, dtype=int)\n        if self.augment:\n            pose, mask, duration, timestamps, dropped = augment_timed_pose(pose, mask, duration, timestamps, self.rng)\n        features = timed_features(pose, mask, duration, timestamps, self.mode)\n        features[:5, :, dropped] = 0\n        return torch.from_numpy(features), torch.tensor(int(self.data["labels"][index]), dtype=torch.long)\n\n\ndef make_loader(data, indices, variant, config, augment, seed):\n    return DataLoader(TimedPoseDataset(data, indices, variant["timing_mode"], augment, seed),\n                      batch_size=config["batch_size"], shuffle=augment, num_workers=0,\n                      generator=torch.Generator().manual_seed(seed))\n', 'research08/training.py': 'import json\nimport platform\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport sklearn\nimport torch\nfrom torch import nn\n\nfrom research07.features import sha256\nfrom research07.models import SkeletonModel, set_seed\nfrom research07.training import metrics, predict, select_thresholds, write_json\nfrom research08.dataset import make_loader\nfrom research08.features import FEATURE_VERSION\n\n\nVARIANTS = {\n    "T0_legacy_dt": {"channels": 8, "attention": True, "fall_multiplier": 2.25, "timing_mode": "legacy"},\n    "T1_interval_dt": {"channels": 8, "attention": True, "fall_multiplier": 2.25, "timing_mode": "interval"},\n}\nPOLICIES = ["fixed", "balanced", "sensitivity"]\n\n\ndef default_config(mode="full"):\n    if mode not in {"full", "smoke"}:\n        raise ValueError("Mode must be full or smoke")\n    return {"mode": mode, "feature_version": FEATURE_VERSION, "variants": list(VARIANTS),\n            "variant_definitions": VARIANTS, "seeds": [42, 142, 242] if mode == "full" else [42],\n            "max_epochs": 50 if mode == "full" else 2, "patience": 10 if mode == "full" else 1,\n            "batch_size": 16, "learning_rate": 0.001, "weight_decay": 0.0001, "dropout": 0.3,\n            "scheduler": None, "allow_cpu_full": False, "target_sensitivity": 0.95,\n            "minimum_specificity": 0.75, "threshold_grid": np.linspace(0.05, 0.95, 37).tolist(),\n            "primary_policy": "fixed", "primary_threshold": 0.5,\n            "epoch_selection": "minimum unweighted inner validation CE; outer refit uses median best epoch",\n            "protocol": "4 outer subjects; 3 inner subjects; matched seeds and augmentation; no all-subject deployment retrain",\n            "timing_note": "frame indices/FPS are nominal timestamps, not measured presentation timestamps",\n            "evaluation_note": "Known outer results informed development; this is not a new blind independent confirmation"}\n\n\ndef fit_model(data, train_indices, validation_indices, variant, config, seed, device, fixed_epochs=None):\n    set_seed(seed)\n    model = SkeletonModel(variant["channels"], variant["attention"], config["dropout"]).to(device)\n    train_loader = make_loader(data, train_indices, variant, config, True, seed)\n    counts = np.bincount(data["labels"][train_indices], minlength=2)\n    if np.any(counts == 0):\n        raise ValueError("Training fold has a missing class")\n    class_weights = len(train_indices) / (2 * counts.astype(np.float32))\n    class_weights[1] *= variant["fall_multiplier"]\n    criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))\n    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])\n    validation_loader = make_loader(data, validation_indices, variant, config, False, seed) if validation_indices is not None else None\n    best_loss, best_state, best_epoch, stale = float("inf"), None, 0, 0\n    history = []\n    for epoch in range(1, (fixed_epochs or config["max_epochs"]) + 1):\n        model.train()\n        total_loss = 0\n        for features, target in train_loader:\n            features, target = features.to(device), target.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            logits, _ = model(features)\n            loss = criterion(logits, target)\n            if not torch.isfinite(loss):\n                raise ValueError("Nonfinite training loss; do not summarize this run")\n            loss.backward()\n            optimizer.step()\n            total_loss += loss.item() * len(target)\n        record = {"epoch": epoch, "train_weighted_loss": total_loss / len(train_indices)}\n        if validation_loader is not None:\n            labels, scores, _ = predict(model, validation_loader, device)\n            scores = np.clip(scores, 1e-7, 1 - 1e-7)\n            validation_loss = float(-(labels * np.log(scores) + (1 - labels) * np.log(1 - scores)).mean())\n            if not np.isfinite(validation_loss):\n                raise ValueError("Nonfinite validation loss")\n            record["validation_unweighted_loss"] = validation_loss\n            if validation_loss < best_loss - 1e-6:\n                best_loss, best_epoch, stale = validation_loss, epoch, 0\n                best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}\n            else:\n                stale += 1\n        history.append(record)\n        if validation_loader is not None and stale >= config["patience"]:\n            break\n    if best_state is not None:\n        model.load_state_dict(best_state)\n    return model, best_epoch if validation_loader is not None else fixed_epochs, history\n\n\ndef require_score_alignment(report, indices, data):\n    if not report.get("complete") or report.get("val_indices") != indices.tolist():\n        raise ValueError("Incomplete/misaligned cached inner run")\n    if report["labels"] != data["labels"][indices].tolist():\n        raise ValueError("Cached validation labels changed")\n    scores = np.asarray(report["scores"], dtype=np.float64)\n    if scores.shape != (len(indices),) or not np.isfinite(scores).all() or np.any((scores < 0) | (scores > 1)):\n        raise ValueError("Invalid cached validation probabilities")\n\n\ndef check_artifacts(report, output):\n    if not report.get("complete") or not report.get("artifacts"):\n        raise ValueError("Outer run not complete")\n    for name, expected_hash in report["artifacts"].items():\n        path = output / name\n        if not path.is_file() or sha256(path) != expected_hash:\n            raise ValueError(f"Missing or corrupt completed artifact: {name}")\n\n\ndef run_neural(data, config, output, device):\n    subjects = sorted(np.unique(data["subjects"]).tolist())\n    expected_runs = len(config["variants"]) * len(config["seeds"]) * len(subjects)\n    completed = 0\n    inner_root = output / "inner_runs"\n    inner_root.mkdir(exist_ok=True)\n    for variant_name in config["variants"]:\n        variant = VARIANTS[variant_name]\n        for seed in config["seeds"]:\n            for outer in subjects:\n                name = f"{variant_name}_seed{seed}_subject{outer}"\n                target = output / f"{name}.json"\n                if target.exists():\n                    check_artifacts(json.loads(target.read_text(encoding="utf-8")), output)\n                    completed += 1\n                    print(f"Resume verified [{completed}/{expected_runs}]: {name}", flush=True)\n                    continue\n                train_indices = np.flatnonzero(data["subjects"] != outer)\n                test_indices = np.flatnonzero(data["subjects"] == outer)\n                inner_scores, inner_labels, inner_epochs, histories, splits = [], [], [], [], []\n                for inner in subjects:\n                    if inner == outer:\n                        continue\n                    fit_indices = np.flatnonzero((data["subjects"] != outer) & (data["subjects"] != inner))\n                    val_indices = np.flatnonzero(data["subjects"] == inner)\n                    if set(fit_indices) & set(val_indices) or set(train_indices) & set(test_indices):\n                        raise ValueError("Subject split overlap")\n                    inner_path = inner_root / f"{name}_inner{inner}.json"\n                    run_seed = seed + outer * 1000 + inner * 10\n                    if inner_path.exists():\n                        report = json.loads(inner_path.read_text(encoding="utf-8"))\n                        require_score_alignment(report, val_indices, data)\n                        if report["train_indices"] != fit_indices.tolist() or report["run_seed"] != run_seed:\n                            raise ValueError("Cached inner split/seed mismatch")\n                    else:\n                        model, best_epoch, history = fit_model(data, fit_indices, val_indices, variant, config, run_seed, device)\n                        labels, scores, _ = predict(model, make_loader(data, val_indices, variant, config, False, run_seed), device)\n                        report = {"complete": True, "train_indices": fit_indices.tolist(), "val_indices": val_indices.tolist(),\n                                  "inner_subject": inner, "run_seed": run_seed, "best_epoch": best_epoch,\n                                  "labels": labels.tolist(), "scores": scores.tolist(), "history": history}\n                        require_score_alignment(report, val_indices, data)\n                        write_json(inner_path, report)\n                        del model\n                    inner_scores.extend(report["scores"])\n                    inner_labels.extend(report["labels"])\n                    inner_epochs.append(report["best_epoch"])\n                    histories.extend([{**row, "inner_subject": inner} for row in report["history"]])\n                    splits.append({"inner_subject": inner, "train_indices": fit_indices.tolist(),\n                                   "val_indices": val_indices.tolist(), "best_epoch": report["best_epoch"]})\n                    print(f"  {name}: inner subject {inner}, best epoch {report[\'best_epoch\']}", flush=True)\n                choices = select_thresholds(np.asarray(inner_labels), np.asarray(inner_scores), config)\n                epochs = max(1, int(np.median(inner_epochs)))\n                model, _, refit_history = fit_model(data, train_indices, None, variant, config, seed + outer * 1000 + 999, device, epochs)\n                labels, scores, attention = predict(model, make_loader(data, test_indices, variant, config, False, seed), device)\n                if not np.array_equal(labels, data["labels"][test_indices]) or not np.isfinite(scores).all():\n                    raise ValueError("Outer prediction alignment/nonfinite output")\n                predictions, fold_metrics = [], []\n                for policy, choice in choices.items():\n                    threshold = choice["threshold"]\n                    predicted = (scores >= threshold).astype(int)\n                    fold_metrics.append({"variant": variant_name, "seed": seed, "subject": outer, "policy": policy,\n                                         "threshold": threshold, "epochs": epochs, "inner_target_met": choice["feasible"], **metrics(labels, predicted)})\n                    for index, label, score, prediction in zip(test_indices, labels, scores, predicted):\n                        predictions.append({"sample_index": int(index), "video_id": str(data["video_ids"][index]),\n                                            "subject": outer, "label": int(label), "score": float(score),\n                                            "prediction": int(prediction), "threshold": threshold,\n                                            "variant": variant_name, "seed": seed, "policy": policy})\n                checkpoint = {"model_state_dict": {key: value.detach().cpu().clone() for key, value in model.state_dict().items()},\n                              "variant": variant, "feature_version": FEATURE_VERSION, "epochs": epochs,\n                              "train_indices": train_indices.tolist(), "test_indices": test_indices.tolist(),\n                              "config": config, "timing_evidence_hash": data["timing_evidence_hash"],\n                              "warning": "Outer-fold research checkpoint; not all-subject deployment model"}\n                checkpoint_path = output / f"{name}.pt"\n                temporary = checkpoint_path.with_suffix(".pending")\n                torch.save(checkpoint, temporary)\n                temporary.replace(checkpoint_path)\n                np.savez_compressed(output / f"{name}_attention.npz", indices=test_indices, attention=attention)\n                pd.DataFrame(histories).to_csv(output / f"{name}_inner_history.csv", index=False)\n                pd.DataFrame(refit_history).to_csv(output / f"{name}_refit_history.csv", index=False)\n                artifact_names = [f"{name}.pt", f"{name}_attention.npz", f"{name}_inner_history.csv", f"{name}_refit_history.csv"]\n                artifacts = {filename: sha256(output / filename) for filename in artifact_names}\n                write_json(target, {"complete": True, "variant": variant_name, "seed": seed, "outer_subject": outer,\n                                    "inner_splits": splits, "inner_labels": inner_labels, "inner_scores": inner_scores,\n                                    "choices": choices, "metrics": fold_metrics, "predictions": predictions,\n                                    "artifacts": artifacts, "parameters": sum(parameter.numel() for parameter in model.parameters())})\n                completed += 1\n                fixed = fold_metrics[0]\n                print(f"Completed [{completed}/{expected_runs}]: {name}; epoch={epochs}; fixed FN={fixed[\'fn\']}, FP={fixed[\'fp\']}", flush=True)\n                del model\n\n\ndef summarize_run(data, config, output):\n    reports = []\n    for variant_name in config["variants"]:\n        for seed in config["seeds"]:\n            for outer in [1, 2, 3, 4]:\n                path = output / f"{variant_name}_seed{seed}_subject{outer}.json"\n                if not path.is_file():\n                    raise ValueError(f"Missing outer run: {path.name}; no final summary")\n                report = json.loads(path.read_text(encoding="utf-8"))\n                check_artifacts(report, output)\n                reports.append(report)\n    predictions = pd.DataFrame([row for report in reports for row in report["predictions"]])\n    folds = pd.DataFrame([row for report in reports for row in report["metrics"]])\n    expected_groups = len(config["variants"]) * len(config["seeds"]) * len(POLICIES)\n    grouped = predictions.groupby(["variant", "seed", "policy"])\n    if grouped.ngroups != expected_groups:\n        raise ValueError("Incomplete prediction policy groups")\n    pooled = []\n    for (variant, seed, policy), group in grouped:\n        group = group.sort_values("sample_index")\n        if not np.array_equal(group["sample_index"], np.arange(160)):\n            raise ValueError("OOF must cover 160 distinct samples exactly once per seed/policy")\n        if not np.array_equal(group["video_id"], data["video_ids"]) or not np.array_equal(group["label"], data["labels"]):\n            raise ValueError("OOF ID/label mismatch")\n        if not np.array_equal(group["subject"], data["subjects"]):\n            raise ValueError("OOF subject mismatch")\n        if not np.array_equal(group["prediction"], (group["score"] >= group["threshold"]).astype(int)):\n            raise ValueError("OOF threshold/prediction mismatch")\n        pooled.append({"variant": variant, "seed": int(seed), "policy": policy, **metrics(group["label"], group["prediction"])})\n    pooled = pd.DataFrame(pooled)\n    metric_columns = ["accuracy", "balanced_accuracy", "sensitivity", "specificity", "precision", "f1", "f2", "false_alarm_rate", "fn", "fp"]\n    merged = pooled[pooled["variant"] == "T0_legacy_dt"].merge(pooled[pooled["variant"] == "T1_interval_dt"], on=["seed", "policy"], suffixes=("_legacy", "_interval"), validate="one_to_one")\n    paired_metrics = merged[["seed", "policy"]].copy()\n    for column in metric_columns:\n        paired_metrics[f"{column}_legacy"] = merged[f"{column}_legacy"]\n        paired_metrics[f"{column}_interval"] = merged[f"{column}_interval"]\n        paired_metrics[f"delta_{column}_interval_minus_legacy"] = merged[f"{column}_interval"] - merged[f"{column}_legacy"]\n    keys = ["seed", "policy", "sample_index", "video_id", "subject", "label"]\n    paired_predictions = predictions[predictions["variant"] == "T0_legacy_dt"].merge(predictions[predictions["variant"] == "T1_interval_dt"], on=keys, suffixes=("_legacy", "_interval"), validate="one_to_one")\n    paired_predictions["legacy_correct"] = paired_predictions["prediction_legacy"] == paired_predictions["label"]\n    paired_predictions["interval_correct"] = paired_predictions["prediction_interval"] == paired_predictions["label"]\n    paired_predictions["error_transition"] = np.select([\n        ~paired_predictions["legacy_correct"] & paired_predictions["interval_correct"],\n        paired_predictions["legacy_correct"] & ~paired_predictions["interval_correct"],\n        ~paired_predictions["legacy_correct"] & ~paired_predictions["interval_correct"]],\n        ["rescued", "new_error", "both_wrong"], default="both_correct")\n    for name, frame in [("predictions", predictions), ("fold_metrics", folds), ("pooled_by_seed", pooled),\n                        ("errors", predictions[predictions["label"] != predictions["prediction"]]),\n                        ("paired_metrics", paired_metrics), ("paired_predictions", paired_predictions)]:\n        frame.to_csv(output / f"{name}.csv", index=False)\n    pooled.groupby(["variant", "policy"])[metric_columns].agg(["mean", "std"]).to_csv(output / "seed_mean_std.csv")\n    paired_metrics.groupby("policy")[[f"delta_{column}_interval_minus_legacy" for column in metric_columns]].agg(["mean", "std"]).to_csv(output / "paired_delta_mean_std.csv")\n    scope = {"complete": True, "mode": config["mode"], "full_training_protocol_completed": config["mode"] == "full",\n             "independent_blind_confirmation": False, "outer_runs": len(reports), "model_fits": len(reports) * 4,\n             "independent_videos": 160, "subjects": 4, "seeds": config["seeds"], "primary_policy": "fixed",\n             "trained_all_subject_deployment_model": False, "production_changed": False,\n             "warning": "Smoke checks software only. Full is development comparison on previously inspected data. Seeds do not create new independent videos."}\n    write_json(output / "completion.json", scope)\n    return pooled\n\n\ndef run_experiment(data, config, output, code_hash):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    if config["variants"] != list(VARIANTS) or config["variant_definitions"] != VARIANTS:\n        raise ValueError("Do not silently change the two preregistered branches")\n    expected_seeds = [42, 142, 242] if config["mode"] == "full" else [42]\n    if config["seeds"] != expected_seeds:\n        raise ValueError("All preregistered seeds are required; do not select the best seed")\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    if device == "cpu" and config["mode"] == "full" and not config["allow_cpu_full"]:\n        raise RuntimeError("Full run requires Kaggle GPU. Local verification uses a separate smoke output.")\n    signature = {"config": config, "inputs": {key: value["sha256"] for key, value in data["provenance"].items()},\n                 "timing_evidence_hash": data["timing_evidence_hash"], "code_hash": code_hash,\n                 "versions": {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__,\n                              "pandas": pd.__version__, "sklearn": sklearn.__version__, "cuda": torch.version.cuda,\n                              "cudnn": torch.backends.cudnn.version()}, "device": device,\n                 "gpu_name": torch.cuda.get_device_name(0) if device == "cuda" else None}\n    lock = output / "locked_protocol.json"\n    if lock.exists() and json.loads(lock.read_text(encoding="utf-8")) != signature:\n        raise ValueError("Configuration/code/input/environment changed. Use a new output directory, never mix old results.")\n    write_json(lock, signature)\n    write_json(output / "input_provenance.json", data["provenance"])\n    write_json(output / "timing_evidence.json", data["timing_evidence"])\n    np.savez_compressed(output / "nominal_sample_timing.npz", video_ids=data["video_ids"].astype("U"),\n                        frame_indices=data["frame_indices"], timestamps=data["timestamps"])\n    started = time.monotonic()\n    run_neural(data, config, output, device)\n    summary = summarize_run(data, config, output)\n    write_json(output / "timing.json", {"seconds_this_invocation": time.monotonic() - started, "device": device})\n    return summary\n', 'research08/validate_pipeline.py': 'import argparse\nimport copy\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom research07.features import augment_pose, load_inputs, make_features_v2\nfrom research07.models import PoseDataset, SkeletonModel\nfrom research08.dataset import TimedPoseDataset\nfrom research08.features import UNCHANGED_CHANNELS, attach_nominal_timing, augment_timed_pose, timed_features\n\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef check_pipeline(data):\n    legacy_max_difference = 0.0\n    changed_videos = 0\n    for index in range(len(data["labels"])):\n        arguments = (data["poses"][index], data["masks"][index], data["durations"][index])\n        old = make_features_v2(*arguments)\n        legacy = timed_features(*arguments, data["timestamps"][index], "legacy")\n        interval = timed_features(*arguments, data["timestamps"][index], "interval")\n        np.testing.assert_array_equal(old, legacy)\n        np.testing.assert_array_equal(old[UNCHANGED_CHANNELS], interval[UNCHANGED_CHANNELS])\n        np.testing.assert_array_equal(interval[3:5, 0], np.zeros((2, 33), dtype=np.float32))\n        legacy_max_difference = max(legacy_max_difference, float(np.max(np.abs(old - legacy))))\n        changed_videos += int(not np.array_equal(old[3:5], interval[3:5]))\n    constant = np.zeros((32, 33, 4), dtype=np.float32)\n    constant[:, [11, 12], 1] = -1\n    constant[..., 3] = 1\n    positions = np.linspace(0, 87, 32).round().astype(int)\n    timestamps = positions / 30\n    constant[:, 15, 0] = timestamps\n    mask = np.ones(32, dtype=bool)\n    corrected = timed_features(constant, mask, 88 / 30, timestamps, "interval")\n    np.testing.assert_allclose(corrected[3, 1:, 15], 1, rtol=1e-5, atol=1e-5)\n    for invalid in [np.zeros(32), np.full(32, np.nan), np.arange(31), -np.arange(32)]:\n        try:\n            timed_features(constant, mask, 88 / 30, invalid, "interval")\n        except ValueError:\n            continue\n        raise AssertionError("Invalid timestamps accepted")\n    uniform = np.arange(32, dtype=np.float64) * (float(data["durations"][0]) / 31)\n    legacy = make_features_v2(data["poses"][0], data["masks"][0], data["durations"][0])\n    uniform_features = timed_features(data["poses"][0], data["masks"][0], data["durations"][0], uniform, "interval")\n    np.testing.assert_allclose(legacy, uniform_features, rtol=1e-5, atol=1e-5)\n    crop_exercised = False\n    for seed in range(30):\n        rng_old, rng_new = np.random.default_rng(seed), np.random.default_rng(seed)\n        original = augment_pose(data["poses"][0], data["masks"][0], data["durations"][0], rng_old)\n        augmented = augment_timed_pose(data["poses"][0], data["masks"][0], data["durations"][0], data["timestamps"][0], rng_new)\n        for old_value, new_value in zip(original, [augmented[0], augmented[1], augmented[2], augmented[4]]):\n            np.testing.assert_array_equal(old_value, new_value)\n        if rng_old.bit_generator.state != rng_new.bit_generator.state:\n            raise AssertionError("Augmentation changed the RNG stream")\n        np.testing.assert_array_equal(augmented[0][..., 3], original[0][..., 3])\n        baseline = make_features_v2(*augmented[:3])\n        candidate = timed_features(*augmented[:3], augmented[3], "interval")\n        np.testing.assert_array_equal(baseline[UNCHANGED_CHANNELS], candidate[UNCHANGED_CHANNELS])\n        crop_exercised |= not np.array_equal(data["timestamps"][0], augmented[3])\n        probe = np.random.default_rng(seed)\n        if probe.random() < 0.5:\n            length = int(probe.integers(26, 33))\n            start = int(probe.integers(0, 33 - length))\n            expected_times = np.interp(np.linspace(start, start + length - 1, 32), np.arange(32), data["timestamps"][0])\n            np.testing.assert_array_equal(augmented[3], expected_times)\n    if not crop_exercised:\n        raise AssertionError("Temporal crop was not exercised")\n    indices = np.array([0, 31, 32, 79, 122, 159])\n    saved_poses = data["poses"][indices].copy()\n    saved_masks = data["masks"][indices].copy()\n    saved_times = data["timestamps"][indices].copy()\n    for augment in [False, True]:\n        old_dataset = PoseDataset(data, indices, 8, augment, 42)\n        legacy_dataset = TimedPoseDataset(data, indices, "legacy", augment, 42)\n        interval_dataset = TimedPoseDataset(data, indices, "interval", augment, 42)\n        for offset in range(len(indices)):\n            old_features, old_label = old_dataset[offset]\n            legacy_features, legacy_label = legacy_dataset[offset]\n            interval_features, interval_label = interval_dataset[offset]\n            torch.testing.assert_close(old_features, legacy_features, rtol=0, atol=0)\n            torch.testing.assert_close(old_features[UNCHANGED_CHANNELS], interval_features[UNCHANGED_CHANNELS], rtol=0, atol=0)\n            if old_label != legacy_label or old_label != interval_label:\n                raise AssertionError("Dataset label changed")\n    np.testing.assert_array_equal(data["poses"][indices], saved_poses)\n    np.testing.assert_array_equal(data["masks"][indices], saved_masks)\n    np.testing.assert_array_equal(data["timestamps"][indices], saved_times)\n    model = SkeletonModel(8, True, 0.3).eval()\n    with torch.no_grad():\n        logits, attention = model(torch.stack([TimedPoseDataset(data, indices, "interval", False, 42)[offset][0] for offset in range(2)]))\n    if logits.shape != (2, 2) or attention.shape != (2, 16) or not torch.isfinite(logits).all():\n        raise AssertionError("Model forward shape/nonfinite output")\n    torch.testing.assert_close(attention.sum(dim=1), torch.ones(2), rtol=1e-5, atol=1e-5)\n    altered = copy.deepcopy(data)\n    altered["manifest"].loc[0, "frame_count"] += 1\n    try:\n        attach_nominal_timing(altered, data["timing_evidence"])\n    except ValueError:\n        pass\n    else:\n        raise AssertionError("Changed manifest accepted")\n    return {"samples_checked": len(data["labels"]), "legacy_feature_max_absolute_difference": legacy_max_difference,\n            "unchanged_six_channels_exact": True, "changed_velocity_videos": changed_videos,\n            "augmentation_parity_seeds": 30, "temporal_crop_timestamps_checked": True,\n            "known_synthetic_velocity_checked": True, "model_forward_checked": True,\n            "input_arrays_unchanged": True, "scientific_result": False}\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--input-root", type=Path, default=ROOT / "Kaggle_kiem_tra_2026-09-25")\n    parser.add_argument("--evidence", type=Path, default=ROOT / "output/research08/timing_evidence.json")\n    arguments = parser.parse_args()\n    torch.set_num_threads(2)\n    data = load_inputs(arguments.input_root)\n    attach_nominal_timing(data, json.loads(arguments.evidence.read_text(encoding="utf-8")))\n    print(json.dumps(check_pipeline(data), indent=2))\n', 'research08/reporting.py': 'import json\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\nfrom research08.training import VARIANTS\n\n\ndef export_figures(output, config):\n    output = Path(output)\n    pooled = pd.read_csv(output / "pooled_by_seed.csv")\n    fixed = pooled[pooled["policy"] == "fixed"]\n    seeds = config["seeds"]\n    figure, axes = plt.subplots(1, 3, figsize=(13, 4))\n    for axis, metric in zip(axes, ["sensitivity", "specificity", "accuracy"]):\n        for variant_name in VARIANTS:\n            group = fixed[fixed["variant"] == variant_name].set_index("seed").loc[seeds]\n            axis.plot(np.arange(len(seeds)), group[metric] * 100, marker="o", label=variant_name)\n        axis.set_xticks(np.arange(len(seeds)), seeds)\n        axis.set_xlabel("Seed")\n        axis.set_ylabel("Percent")\n        axis.set_title(metric)\n        axis.set_ylim(0, 105)\n        axis.grid(alpha=0.2)\n    axes[0].legend(fontsize=8)\n    scope = "SMOKE CHECK ONLY" if config["mode"] == "smoke" else "Development comparison"\n    figure.suptitle(f"{scope}\\nFixed threshold 0.5 - all seeds, same 160 videos", fontsize=11)\n    figure.tight_layout()\n    figure.savefig(output / "fixed_metrics_by_seed.png", dpi=160, bbox_inches="tight")\n    plt.close(figure)\n    figure, axes = plt.subplots(2, len(seeds), figsize=(4 * len(seeds), 7), squeeze=False)\n    for row, variant_name in enumerate(VARIANTS):\n        for column, seed in enumerate(seeds):\n            record = fixed[(fixed["variant"] == variant_name) & (fixed["seed"] == seed)].iloc[0]\n            values = np.array([[record["tn"], record["fp"]], [record["fn"], record["tp"]]], dtype=int)\n            axis = axes[row, column]\n            axis.imshow(values, cmap="Blues", vmin=0, vmax=81)\n            for row_index in range(2):\n                for column_index in range(2):\n                    axis.text(column_index, row_index, str(values[row_index, column_index]), ha="center", va="center",\n                              color="white" if values[row_index, column_index] > 40 else "black", fontsize=14)\n            axis.set_xticks([0, 1], ["ADL", "Fall"])\n            axis.set_yticks([0, 1], ["ADL", "Fall"])\n            axis.set_xlabel("Predicted")\n            axis.set_ylabel("True")\n            axis.set_title(f"{variant_name} | seed {seed}")\n    figure.suptitle(f"{scope} - OOF confusion counts\\nRepeated seeds are not new videos", fontsize=10)\n    figure.tight_layout()\n    figure.savefig(output / "fixed_confusion_matrices.png", dpi=160, bbox_inches="tight")\n    plt.close(figure)\n    selections = []\n    for variant_name in VARIANTS:\n        for seed in seeds:\n            for outer in [1, 2, 3, 4]:\n                name = f"{variant_name}_seed{seed}_subject{outer}"\n                report = json.loads((output / f"{name}.json").read_text(encoding="utf-8"))\n                for split in report["inner_splits"]:\n                    inner = split["inner_subject"]\n                    history = json.loads((output / "inner_runs" / f"{name}_inner{inner}.json").read_text(encoding="utf-8"))["history"]\n                    selections.append({"variant": variant_name, "seed": seed, "outer_subject": outer,\n                                       "inner_subject": inner, "best_epoch": split["best_epoch"],\n                                       "epochs_run": len(history), "hit_max_epochs": len(history) == config["max_epochs"],\n                                       "selected_at_max_epochs": split["best_epoch"] == config["max_epochs"]})\n    selection_frame = pd.DataFrame(selections)\n    selection_frame.to_csv(output / "inner_epoch_selection.csv", index=False)\n    return {"figures": ["fixed_metrics_by_seed.png", "fixed_confusion_matrices.png"],\n            "inner_fits": len(selection_frame), "inner_fits_hitting_epoch_limit": int(selection_frame["hit_max_epochs"].sum()),\n            "inner_fits_selecting_epoch_limit": int(selection_frame["selected_at_max_epochs"].sum())}\n'}
CODE_HASH = 'e994fe4b09b5f6f16a1d20c336397803cf0855370066bdec6501d91255a76593'
TIMING_EVIDENCE = {'version': '08-decoded-frame-evidence-v1', 'verified_date': '2026-10-05', 'samples': 160, 'input_hashes': {'pose': '598638f33066c4b6487dbaa1ccbfcc98a96143f8deae94354278f635324633b4', 'features': '4a255c925adbcfb8364eb413dd492c72d2a0cfcdd5074082f470b0aa836e2848', 'manifest': '5c41ac710d580a648d6768c5d7ceb2a5b95767f23069519ed5b98737e59417f2', 'folds': '774f0ac28b8342e9c8080af075fcda8227ac6a036124514f6c09f0a375b4fbba'}, 'sampling_rule': 'np.linspace(0, decoded_frame_count - 1, 32).round().astype(int)', 'source_notebook_sha256': '5b6945679a6fcf568e493991c91d975416d7d1bf075de03667aeb0d2839b8587', 'opencv_version': '4.11.0', 'source_check': 'Every source frame decoded locally; count and nominal FPS match notebook 01 manifest', 'timing_type': 'nominal frame_index / FPS, not original presentation timestamps', 'limitations': 'Source-count agreement and the saved sampling rule do not verify joint accuracy or variable-frame-rate timestamps', 'rows': [{'video_id': 'S1_ADL_01', 'sample_index': 0, 'frame_count': 245, 'decoded_frame_count': 245, 'fps': 29.61249929494049, 'frame_indices': [0, 8, 16, 24, 31, 39, 47, 55, 63, 71, 79, 87, 94, 102, 110, 118, 126, 134, 142, 150, 157, 165, 173, 181, 189, 197, 205, 213, 220, 228, 236, 244]}, {'video_id': 'S1_ADL_02', 'sample_index': 1, 'frame_count': 198, 'decoded_frame_count': 198, 'fps': 29.598477223124682, 'frame_indices': [0, 6, 13, 19, 25, 32, 38, 44, 51, 57, 64, 70, 76, 83, 89, 95, 102, 108, 114, 121, 127, 133, 140, 146, 153, 159, 165, 172, 178, 184, 191, 197]}, {'video_id': 'S1_ADL_03', 'sample_index': 2, 'frame_count': 199, 'decoded_frame_count': 199, 'fps': 29.74840918264128, 'frame_indices': [0, 6, 13, 19, 26, 32, 38, 45, 51, 57, 64, 70, 77, 83, 89, 96, 102, 109, 115, 121, 128, 134, 141, 147, 153, 160, 166, 172, 179, 185, 192, 198]}, {'video_id': 'S1_ADL_04', 'sample_index': 3, 'frame_count': 170, 'decoded_frame_count': 170, 'fps': 29.263422443323638, 'frame_indices': [0, 5, 11, 16, 22, 27, 33, 38, 44, 49, 55, 60, 65, 71, 76, 82, 87, 93, 98, 104, 109, 114, 120, 125, 131, 136, 142, 147, 153, 158, 164, 169]}, {'video_id': 'S1_ADL_05', 'sample_index': 4, 'frame_count': 314, 'decoded_frame_count': 314, 'fps': 29.775451373084508, 'frame_indices': [0, 10, 20, 30, 40, 50, 61, 71, 81, 91, 101, 111, 121, 131, 141, 151, 162, 172, 182, 192, 202, 212, 222, 232, 242, 252, 263, 273, 283, 293, 303, 313]}, {'video_id': 'S1_ADL_06', 'sample_index': 5, 'frame_count': 364, 'decoded_frame_count': 364, 'fps': 29.773833381047808, 'frame_indices': [0, 12, 23, 35, 47, 59, 70, 82, 94, 105, 117, 129, 141, 152, 164, 176, 187, 199, 211, 222, 234, 246, 258, 269, 281, 293, 304, 316, 328, 340, 351, 363]}, {'video_id': 'S1_ADL_07', 'sample_index': 6, 'frame_count': 224, 'decoded_frame_count': 224, 'fps': 29.71912770822185, 'frame_indices': [0, 7, 14, 22, 29, 36, 43, 50, 58, 65, 72, 79, 86, 94, 101, 108, 115, 122, 129, 137, 144, 151, 158, 165, 173, 180, 187, 194, 201, 209, 216, 223]}, {'video_id': 'S1_ADL_08', 'sample_index': 7, 'frame_count': 187, 'decoded_frame_count': 187, 'fps': 29.73240832507433, 'frame_indices': [0, 6, 12, 18, 24, 30, 36, 42, 48, 54, 60, 66, 72, 78, 84, 90, 96, 102, 108, 114, 120, 126, 132, 138, 144, 150, 156, 162, 168, 174, 180, 186]}, {'video_id': 'S1_ADL_09', 'sample_index': 8, 'frame_count': 147, 'decoded_frame_count': 147, 'fps': 29.43768023069529, 'frame_indices': [0, 5, 9, 14, 19, 24, 28, 33, 38, 42, 47, 52, 57, 61, 66, 71, 75, 80, 85, 89, 94, 99, 104, 108, 113, 118, 122, 127, 132, 137, 141, 146]}, {'video_id': 'S1_ADL_10', 'sample_index': 9, 'frame_count': 115, 'decoded_frame_count': 115, 'fps': 29.32750750189141, 'frame_indices': [0, 4, 7, 11, 15, 18, 22, 26, 29, 33, 37, 40, 44, 48, 51, 55, 59, 63, 66, 70, 74, 77, 81, 85, 88, 92, 96, 99, 103, 107, 110, 114]}, {'video_id': 'S1_ADL_11', 'sample_index': 10, 'frame_count': 277, 'decoded_frame_count': 277, 'fps': 29.741666249114193, 'frame_indices': [0, 9, 18, 27, 36, 45, 53, 62, 71, 80, 89, 98, 107, 116, 125, 134, 142, 151, 160, 169, 178, 187, 196, 205, 214, 223, 231, 240, 249, 258, 267, 276]}, {'video_id': 'S1_ADL_12', 'sample_index': 11, 'frame_count': 218, 'decoded_frame_count': 218, 'fps': 29.742684845783725, 'frame_indices': [0, 7, 14, 21, 28, 35, 42, 49, 56, 63, 70, 77, 84, 91, 98, 105, 112, 119, 126, 133, 140, 147, 154, 161, 168, 175, 182, 189, 196, 203, 210, 217]}, {'video_id': 'S1_ADL_13', 'sample_index': 12, 'frame_count': 187, 'decoded_frame_count': 187, 'fps': 29.656178634864247, 'frame_indices': [0, 6, 12, 18, 24, 30, 36, 42, 48, 54, 60, 66, 72, 78, 84, 90, 96, 102, 108, 114, 120, 126, 132, 138, 144, 150, 156, 162, 168, 174, 180, 186]}, {'video_id': 'S1_ADL_14', 'sample_index': 13, 'frame_count': 173, 'decoded_frame_count': 173, 'fps': 29.533944118818642, 'frame_indices': [0, 6, 11, 17, 22, 28, 33, 39, 44, 50, 55, 61, 67, 72, 78, 83, 89, 94, 100, 105, 111, 117, 122, 128, 133, 139, 144, 150, 155, 161, 166, 172]}, {'video_id': 'S1_ADL_15', 'sample_index': 14, 'frame_count': 212, 'decoded_frame_count': 212, 'fps': 29.570804874533305, 'frame_indices': [0, 7, 14, 20, 27, 34, 41, 48, 54, 61, 68, 75, 82, 88, 95, 102, 109, 116, 123, 129, 136, 143, 150, 157, 163, 170, 177, 184, 191, 197, 204, 211]}, {'video_id': 'S1_ADL_16', 'sample_index': 15, 'frame_count': 104, 'decoded_frame_count': 104, 'fps': 29.5337081842449, 'frame_indices': [0, 3, 7, 10, 13, 17, 20, 23, 27, 30, 33, 37, 40, 43, 47, 50, 53, 56, 60, 63, 66, 70, 73, 76, 80, 83, 86, 90, 93, 96, 100, 103]}, {'video_id': 'S1_FALL_01', 'sample_index': 16, 'frame_count': 203, 'decoded_frame_count': 203, 'fps': 29.567986949302313, 'frame_indices': [0, 7, 13, 20, 26, 33, 39, 46, 52, 59, 65, 72, 78, 85, 91, 98, 104, 111, 117, 124, 130, 137, 143, 150, 156, 163, 169, 176, 182, 189, 195, 202]}, {'video_id': 'S1_FALL_02', 'sample_index': 17, 'frame_count': 189, 'decoded_frame_count': 189, 'fps': 29.67265876442421, 'frame_indices': [0, 6, 12, 18, 24, 30, 36, 42, 49, 55, 61, 67, 73, 79, 85, 91, 97, 103, 109, 115, 121, 127, 133, 139, 146, 152, 158, 164, 170, 176, 182, 188]}, {'video_id': 'S1_FALL_03', 'sample_index': 18, 'frame_count': 176, 'decoded_frame_count': 176, 'fps': 29.721529532955433, 'frame_indices': [0, 6, 11, 17, 23, 28, 34, 40, 45, 51, 56, 62, 68, 73, 79, 85, 90, 96, 102, 107, 113, 119, 124, 130, 135, 141, 147, 152, 158, 164, 169, 175]}, {'video_id': 'S1_FALL_04', 'sample_index': 19, 'frame_count': 146, 'decoded_frame_count': 146, 'fps': 29.427770947130792, 'frame_indices': [0, 5, 9, 14, 19, 23, 28, 33, 37, 42, 47, 51, 56, 61, 65, 70, 75, 80, 84, 89, 94, 98, 103, 108, 112, 117, 122, 126, 131, 136, 140, 145]}, {'video_id': 'S1_FALL_05', 'sample_index': 20, 'frame_count': 174, 'decoded_frame_count': 174, 'fps': 29.70482959557045, 'frame_indices': [0, 6, 11, 17, 22, 28, 33, 39, 45, 50, 56, 61, 67, 73, 78, 84, 89, 95, 100, 106, 112, 117, 123, 128, 134, 140, 145, 151, 156, 162, 167, 173]}, {'video_id': 'S1_FALL_06', 'sample_index': 21, 'frame_count': 169, 'decoded_frame_count': 169, 'fps': 29.663520890256674, 'frame_indices': [0, 5, 11, 16, 22, 27, 33, 38, 43, 49, 54, 60, 65, 70, 76, 81, 87, 92, 98, 103, 108, 114, 119, 125, 130, 135, 141, 146, 152, 157, 163, 168]}, {'video_id': 'S1_FALL_07', 'sample_index': 22, 'frame_count': 153, 'decoded_frame_count': 153, 'fps': 29.688561172019018, 'frame_indices': [0, 5, 10, 15, 20, 25, 29, 34, 39, 44, 49, 54, 59, 64, 69, 74, 78, 83, 88, 93, 98, 103, 108, 113, 118, 123, 127, 132, 137, 142, 147, 152]}, {'video_id': 'S1_FALL_08', 'sample_index': 23, 'frame_count': 129, 'decoded_frame_count': 129, 'fps': 29.631784874773167, 'frame_indices': [0, 4, 8, 12, 17, 21, 25, 29, 33, 37, 41, 45, 50, 54, 58, 62, 66, 70, 74, 78, 83, 87, 91, 95, 99, 103, 107, 111, 116, 120, 124, 128]}, {'video_id': 'S1_FALL_09', 'sample_index': 24, 'frame_count': 131, 'decoded_frame_count': 131, 'fps': 29.23106674798804, 'frame_indices': [0, 4, 8, 13, 17, 21, 25, 29, 34, 38, 42, 46, 50, 55, 59, 63, 67, 71, 75, 80, 84, 88, 92, 96, 101, 105, 109, 113, 117, 122, 126, 130]}, {'video_id': 'S1_FALL_10', 'sample_index': 25, 'frame_count': 133, 'decoded_frame_count': 133, 'fps': 29.57264197091653, 'frame_indices': [0, 4, 9, 13, 17, 21, 26, 30, 34, 38, 43, 47, 51, 55, 60, 64, 68, 72, 77, 81, 85, 89, 94, 98, 102, 106, 111, 115, 119, 123, 128, 132]}, {'video_id': 'S1_FALL_11', 'sample_index': 26, 'frame_count': 181, 'decoded_frame_count': 181, 'fps': 29.684783678289108, 'frame_indices': [0, 6, 12, 17, 23, 29, 35, 41, 46, 52, 58, 64, 70, 75, 81, 87, 93, 99, 105, 110, 116, 122, 128, 134, 139, 145, 151, 157, 163, 168, 174, 180]}, {'video_id': 'S1_FALL_12', 'sample_index': 27, 'frame_count': 200, 'decoded_frame_count': 200, 'fps': 29.268007141393742, 'frame_indices': [0, 6, 13, 19, 26, 32, 39, 45, 51, 58, 64, 71, 77, 83, 90, 96, 103, 109, 116, 122, 128, 135, 141, 148, 154, 160, 167, 173, 180, 186, 193, 199]}, {'video_id': 'S1_FALL_13', 'sample_index': 28, 'frame_count': 128, 'decoded_frame_count': 128, 'fps': 29.620487503856836, 'frame_indices': [0, 4, 8, 12, 16, 20, 25, 29, 33, 37, 41, 45, 49, 53, 57, 61, 66, 70, 74, 78, 82, 86, 90, 94, 98, 102, 107, 111, 115, 119, 123, 127]}, {'video_id': 'S1_FALL_14', 'sample_index': 29, 'frame_count': 169, 'decoded_frame_count': 169, 'fps': 29.663000234027614, 'frame_indices': [0, 5, 11, 16, 22, 27, 33, 38, 43, 49, 54, 60, 65, 70, 76, 81, 87, 92, 98, 103, 108, 114, 119, 125, 130, 135, 141, 146, 152, 157, 163, 168]}, {'video_id': 'S1_FALL_15', 'sample_index': 30, 'frame_count': 178, 'decoded_frame_count': 178, 'fps': 29.658922392486406, 'frame_indices': [0, 6, 11, 17, 23, 29, 34, 40, 46, 51, 57, 63, 69, 74, 80, 86, 91, 97, 103, 108, 114, 120, 126, 131, 137, 143, 148, 154, 160, 166, 171, 177]}, {'video_id': 'S1_FALL_16', 'sample_index': 31, 'frame_count': 181, 'decoded_frame_count': 181, 'fps': 29.685270529578666, 'frame_indices': [0, 6, 12, 17, 23, 29, 35, 41, 46, 52, 58, 64, 70, 75, 81, 87, 93, 99, 105, 110, 116, 122, 128, 134, 139, 145, 151, 157, 163, 168, 174, 180]}, {'video_id': 'S2_ADL_01', 'sample_index': 32, 'frame_count': 340, 'decoded_frame_count': 340, 'fps': 29.841139814516836, 'frame_indices': [0, 11, 22, 33, 44, 55, 66, 77, 87, 98, 109, 120, 131, 142, 153, 164, 175, 186, 197, 208, 219, 230, 241, 252, 262, 273, 284, 295, 306, 317, 328, 339]}, {'video_id': 'S2_ADL_02', 'sample_index': 33, 'frame_count': 352, 'decoded_frame_count': 352, 'fps': 29.84720267268133, 'frame_indices': [0, 11, 23, 34, 45, 57, 68, 79, 91, 102, 113, 125, 136, 147, 159, 170, 181, 192, 204, 215, 226, 238, 249, 260, 272, 283, 294, 306, 317, 328, 340, 351]}, {'video_id': 'S2_ADL_03', 'sample_index': 34, 'frame_count': 356, 'decoded_frame_count': 356, 'fps': 29.822405897464538, 'frame_indices': [0, 11, 23, 34, 46, 57, 69, 80, 92, 103, 115, 126, 137, 149, 160, 172, 183, 195, 206, 218, 229, 240, 252, 263, 275, 286, 298, 309, 321, 332, 344, 355]}, {'video_id': 'S2_ADL_04', 'sample_index': 35, 'frame_count': 234, 'decoded_frame_count': 234, 'fps': 29.841102505462366, 'frame_indices': [0, 8, 15, 23, 30, 38, 45, 53, 60, 68, 75, 83, 90, 98, 105, 113, 120, 128, 135, 143, 150, 158, 165, 173, 180, 188, 195, 203, 210, 218, 225, 233]}, {'video_id': 'S2_ADL_05', 'sample_index': 36, 'frame_count': 323, 'decoded_frame_count': 323, 'fps': 29.85893272033674, 'frame_indices': [0, 10, 21, 31, 42, 52, 62, 73, 83, 93, 104, 114, 125, 135, 145, 156, 166, 177, 187, 197, 208, 218, 229, 239, 249, 260, 270, 280, 291, 301, 312, 322]}, {'video_id': 'S2_ADL_06', 'sample_index': 37, 'frame_count': 321, 'decoded_frame_count': 321, 'fps': 29.85102386222032, 'frame_indices': [0, 10, 21, 31, 41, 52, 62, 72, 83, 93, 103, 114, 124, 134, 145, 155, 165, 175, 186, 196, 206, 217, 227, 237, 248, 258, 268, 279, 289, 299, 310, 320]}, {'video_id': 'S2_ADL_07', 'sample_index': 38, 'frame_count': 388, 'decoded_frame_count': 388, 'fps': 29.86131420567365, 'frame_indices': [0, 12, 25, 37, 50, 62, 75, 87, 100, 112, 125, 137, 150, 162, 175, 187, 200, 212, 225, 237, 250, 262, 275, 287, 300, 312, 325, 337, 350, 362, 375, 387]}, {'video_id': 'S2_ADL_08', 'sample_index': 39, 'frame_count': 408, 'decoded_frame_count': 408, 'fps': 29.85598876009835, 'frame_indices': [0, 13, 26, 39, 53, 66, 79, 92, 105, 118, 131, 144, 158, 171, 184, 197, 210, 223, 236, 249, 263, 276, 289, 302, 315, 328, 341, 354, 368, 381, 394, 407]}, {'video_id': 'S2_ADL_09', 'sample_index': 40, 'frame_count': 327, 'decoded_frame_count': 327, 'fps': 29.83449609809801, 'frame_indices': [0, 11, 21, 32, 42, 53, 63, 74, 84, 95, 105, 116, 126, 137, 147, 158, 168, 179, 189, 200, 210, 221, 231, 242, 252, 263, 273, 284, 294, 305, 315, 326]}, {'video_id': 'S2_ADL_10', 'sample_index': 41, 'frame_count': 226, 'decoded_frame_count': 226, 'fps': 29.86244775172766, 'frame_indices': [0, 7, 15, 22, 29, 36, 44, 51, 58, 65, 73, 80, 87, 94, 102, 109, 116, 123, 131, 138, 145, 152, 160, 167, 174, 181, 189, 196, 203, 210, 218, 225]}, {'video_id': 'S2_ADL_11', 'sample_index': 42, 'frame_count': 298, 'decoded_frame_count': 298, 'fps': 29.84387048961974, 'frame_indices': [0, 10, 19, 29, 38, 48, 57, 67, 77, 86, 96, 105, 115, 125, 134, 144, 153, 163, 172, 182, 192, 201, 211, 220, 230, 240, 249, 259, 268, 278, 287, 297]}, {'video_id': 'S2_ADL_12', 'sample_index': 43, 'frame_count': 227, 'decoded_frame_count': 227, 'fps': 29.862396183226192, 'frame_indices': [0, 7, 15, 22, 29, 36, 44, 51, 58, 66, 73, 80, 87, 95, 102, 109, 117, 124, 131, 139, 146, 153, 160, 168, 175, 182, 190, 197, 204, 211, 219, 226]}, {'video_id': 'S2_ADL_13', 'sample_index': 44, 'frame_count': 145, 'decoded_frame_count': 145, 'fps': 29.907390219252107, 'frame_indices': [0, 5, 9, 14, 19, 23, 28, 33, 37, 42, 46, 51, 56, 60, 65, 70, 74, 79, 84, 88, 93, 98, 102, 107, 111, 116, 121, 125, 130, 135, 139, 144]}, {'video_id': 'S2_ADL_14', 'sample_index': 45, 'frame_count': 166, 'decoded_frame_count': 166, 'fps': 15.012344488088097, 'frame_indices': [0, 5, 11, 16, 21, 27, 32, 37, 43, 48, 53, 59, 64, 69, 75, 80, 85, 90, 96, 101, 106, 112, 117, 122, 128, 133, 138, 144, 149, 154, 160, 165]}, {'video_id': 'S2_ADL_15', 'sample_index': 46, 'frame_count': 179, 'decoded_frame_count': 179, 'fps': 15.015015015015015, 'frame_indices': [0, 6, 11, 17, 23, 29, 34, 40, 46, 52, 57, 63, 69, 75, 80, 86, 92, 98, 103, 109, 115, 121, 126, 132, 138, 144, 149, 155, 161, 167, 172, 178]}, {'video_id': 'S2_ADL_16', 'sample_index': 47, 'frame_count': 173, 'decoded_frame_count': 173, 'fps': 14.994626203325975, 'frame_indices': [0, 6, 11, 17, 22, 28, 33, 39, 44, 50, 55, 61, 67, 72, 78, 83, 89, 94, 100, 105, 111, 117, 122, 128, 133, 139, 144, 150, 155, 161, 166, 172]}, {'video_id': 'S2_ADL_17', 'sample_index': 48, 'frame_count': 175, 'decoded_frame_count': 175, 'fps': 15.001414419073798, 'frame_indices': [0, 6, 11, 17, 22, 28, 34, 39, 45, 51, 56, 62, 67, 73, 79, 84, 90, 95, 101, 107, 112, 118, 123, 129, 135, 140, 146, 152, 157, 163, 168, 174]}, {'video_id': 'S2_ADL_18', 'sample_index': 49, 'frame_count': 175, 'decoded_frame_count': 175, 'fps': 15.001714481655046, 'frame_indices': [0, 6, 11, 17, 22, 28, 34, 39, 45, 51, 56, 62, 67, 73, 79, 84, 90, 95, 101, 107, 112, 118, 123, 129, 135, 140, 146, 152, 157, 163, 168, 174]}, {'video_id': 'S2_ADL_19', 'sample_index': 50, 'frame_count': 170, 'decoded_frame_count': 170, 'fps': 15.00520768972761, 'frame_indices': [0, 5, 11, 16, 22, 27, 33, 38, 44, 49, 55, 60, 65, 71, 76, 82, 87, 93, 98, 104, 109, 114, 120, 125, 131, 136, 142, 147, 153, 158, 164, 169]}, {'video_id': 'S2_ADL_20', 'sample_index': 51, 'frame_count': 175, 'decoded_frame_count': 175, 'fps': 15.001628748264098, 'frame_indices': [0, 6, 11, 17, 22, 28, 34, 39, 45, 51, 56, 62, 67, 73, 79, 84, 90, 95, 101, 107, 112, 118, 123, 129, 135, 140, 146, 152, 157, 163, 168, 174]}, {'video_id': 'S2_ADL_21', 'sample_index': 52, 'frame_count': 158, 'decoded_frame_count': 158, 'fps': 15.005698366468279, 'frame_indices': [0, 5, 10, 15, 20, 25, 30, 35, 41, 46, 51, 56, 61, 66, 71, 76, 81, 86, 91, 96, 101, 106, 111, 116, 122, 127, 132, 137, 142, 147, 152, 157]}, {'video_id': 'S2_ADL_22', 'sample_index': 53, 'frame_count': 182, 'decoded_frame_count': 182, 'fps': 15.004987921809173, 'frame_indices': [0, 6, 12, 18, 23, 29, 35, 41, 47, 53, 58, 64, 70, 76, 82, 88, 93, 99, 105, 111, 117, 123, 128, 134, 140, 146, 152, 158, 163, 169, 175, 181]}, {'video_id': 'S2_ADL_23', 'sample_index': 54, 'frame_count': 109, 'decoded_frame_count': 109, 'fps': 14.971910498193756, 'frame_indices': [0, 3, 7, 10, 14, 17, 21, 24, 28, 31, 35, 38, 42, 45, 49, 52, 56, 59, 63, 66, 70, 73, 77, 80, 84, 87, 91, 94, 98, 101, 105, 108]}, {'video_id': 'S2_FALL_01', 'sample_index': 55, 'frame_count': 190, 'decoded_frame_count': 190, 'fps': 29.836214883560245, 'frame_indices': [0, 6, 12, 18, 24, 30, 37, 43, 49, 55, 61, 67, 73, 79, 85, 91, 98, 104, 110, 116, 122, 128, 134, 140, 146, 152, 159, 165, 171, 177, 183, 189]}, {'video_id': 'S2_FALL_02', 'sample_index': 56, 'frame_count': 252, 'decoded_frame_count': 252, 'fps': 29.829898554670393, 'frame_indices': [0, 8, 16, 24, 32, 40, 49, 57, 65, 73, 81, 89, 97, 105, 113, 121, 130, 138, 146, 154, 162, 170, 178, 186, 194, 202, 211, 219, 227, 235, 243, 251]}, {'video_id': 'S2_FALL_03', 'sample_index': 57, 'frame_count': 210, 'decoded_frame_count': 210, 'fps': 29.822908727697907, 'frame_indices': [0, 7, 13, 20, 27, 34, 40, 47, 54, 61, 67, 74, 81, 88, 94, 101, 108, 115, 121, 128, 135, 142, 148, 155, 162, 169, 175, 182, 189, 196, 202, 209]}, {'video_id': 'S2_FALL_04', 'sample_index': 58, 'frame_count': 305, 'decoded_frame_count': 305, 'fps': 29.87374669021741, 'frame_indices': [0, 10, 20, 29, 39, 49, 59, 69, 78, 88, 98, 108, 118, 127, 137, 147, 157, 167, 177, 186, 196, 206, 216, 226, 235, 245, 255, 265, 275, 284, 294, 304]}, {'video_id': 'S2_FALL_05', 'sample_index': 59, 'frame_count': 135, 'decoded_frame_count': 135, 'fps': 29.813609728806572, 'frame_indices': [0, 4, 9, 13, 17, 22, 26, 30, 35, 39, 43, 48, 52, 56, 61, 65, 69, 73, 78, 82, 86, 91, 95, 99, 104, 108, 112, 117, 121, 125, 130, 134]}, {'video_id': 'S2_FALL_06', 'sample_index': 60, 'frame_count': 194, 'decoded_frame_count': 194, 'fps': 29.864225531347174, 'frame_indices': [0, 6, 12, 19, 25, 31, 37, 44, 50, 56, 62, 68, 75, 81, 87, 93, 100, 106, 112, 118, 125, 131, 137, 143, 149, 156, 162, 168, 174, 181, 187, 193]}, {'video_id': 'S2_FALL_07', 'sample_index': 61, 'frame_count': 135, 'decoded_frame_count': 135, 'fps': 29.81404867419502, 'frame_indices': [0, 4, 9, 13, 17, 22, 26, 30, 35, 39, 43, 48, 52, 56, 61, 65, 69, 73, 78, 82, 86, 91, 95, 99, 104, 108, 112, 117, 121, 125, 130, 134]}, {'video_id': 'S2_FALL_08', 'sample_index': 62, 'frame_count': 181, 'decoded_frame_count': 181, 'fps': 29.846644313747046, 'frame_indices': [0, 6, 12, 17, 23, 29, 35, 41, 46, 52, 58, 64, 70, 75, 81, 87, 93, 99, 105, 110, 116, 122, 128, 134, 139, 145, 151, 157, 163, 168, 174, 180]}, {'video_id': 'S2_FALL_09', 'sample_index': 63, 'frame_count': 167, 'decoded_frame_count': 167, 'fps': 29.820541055325734, 'frame_indices': [0, 5, 11, 16, 21, 27, 32, 37, 43, 48, 54, 59, 64, 70, 75, 80, 86, 91, 96, 102, 107, 112, 118, 123, 129, 134, 139, 145, 150, 155, 161, 166]}, {'video_id': 'S2_FALL_10', 'sample_index': 64, 'frame_count': 167, 'decoded_frame_count': 167, 'fps': 14.99438832772166, 'frame_indices': [0, 5, 11, 16, 21, 27, 32, 37, 43, 48, 54, 59, 64, 70, 75, 80, 86, 91, 96, 102, 107, 112, 118, 123, 129, 134, 139, 145, 150, 155, 161, 166]}, {'video_id': 'S2_FALL_11', 'sample_index': 65, 'frame_count': 159, 'decoded_frame_count': 159, 'fps': 15.009439899307742, 'frame_indices': [0, 5, 10, 15, 20, 25, 31, 36, 41, 46, 51, 56, 61, 66, 71, 76, 82, 87, 92, 97, 102, 107, 112, 117, 122, 127, 133, 138, 143, 148, 153, 158]}, {'video_id': 'S2_FALL_12', 'sample_index': 66, 'frame_count': 184, 'decoded_frame_count': 184, 'fps': 15.011340088436373, 'frame_indices': [0, 6, 12, 18, 24, 30, 35, 41, 47, 53, 59, 65, 71, 77, 83, 89, 94, 100, 106, 112, 118, 124, 130, 136, 142, 148, 153, 159, 165, 171, 177, 183]}, {'video_id': 'S2_FALL_13', 'sample_index': 67, 'frame_count': 133, 'decoded_frame_count': 133, 'fps': 14.950539568345324, 'frame_indices': [0, 4, 9, 13, 17, 21, 26, 30, 34, 38, 43, 47, 51, 55, 60, 64, 68, 72, 77, 81, 85, 89, 94, 98, 102, 106, 111, 115, 119, 123, 128, 132]}, {'video_id': 'S2_FALL_14', 'sample_index': 68, 'frame_count': 168, 'decoded_frame_count': 168, 'fps': 14.998080602780002, 'frame_indices': [0, 5, 11, 16, 22, 27, 32, 38, 43, 48, 54, 59, 65, 70, 75, 81, 86, 92, 97, 102, 108, 113, 119, 124, 129, 135, 140, 145, 151, 156, 162, 167]}, {'video_id': 'S2_FALL_15', 'sample_index': 69, 'frame_count': 348, 'decoded_frame_count': 348, 'fps': 29.832066042022067, 'frame_indices': [0, 11, 22, 34, 45, 56, 67, 78, 90, 101, 112, 123, 134, 146, 157, 168, 179, 190, 201, 213, 224, 235, 246, 257, 269, 280, 291, 302, 313, 325, 336, 347]}, {'video_id': 'S2_FALL_16', 'sample_index': 70, 'frame_count': 120, 'decoded_frame_count': 120, 'fps': 29.869569546314427, 'frame_indices': [0, 4, 8, 12, 15, 19, 23, 27, 31, 35, 38, 42, 46, 50, 54, 58, 61, 65, 69, 73, 77, 81, 84, 88, 92, 96, 100, 104, 107, 111, 115, 119]}, {'video_id': 'S2_FALL_17', 'sample_index': 71, 'frame_count': 194, 'decoded_frame_count': 194, 'fps': 29.858250265492174, 'frame_indices': [0, 6, 12, 19, 25, 31, 37, 44, 50, 56, 62, 68, 75, 81, 87, 93, 100, 106, 112, 118, 125, 131, 137, 143, 149, 156, 162, 168, 174, 181, 187, 193]}, {'video_id': 'S2_FALL_18', 'sample_index': 72, 'frame_count': 345, 'decoded_frame_count': 345, 'fps': 29.860994904877586, 'frame_indices': [0, 11, 22, 33, 44, 55, 67, 78, 89, 100, 111, 122, 133, 144, 155, 166, 178, 189, 200, 211, 222, 233, 244, 255, 266, 277, 289, 300, 311, 322, 333, 344]}, {'video_id': 'S2_FALL_19', 'sample_index': 73, 'frame_count': 463, 'decoded_frame_count': 463, 'fps': 29.829975581942236, 'frame_indices': [0, 15, 30, 45, 60, 75, 89, 104, 119, 134, 149, 164, 179, 194, 209, 224, 238, 253, 268, 283, 298, 313, 328, 343, 358, 373, 387, 402, 417, 432, 447, 462]}, {'video_id': 'S2_FALL_20', 'sample_index': 74, 'frame_count': 342, 'decoded_frame_count': 342, 'fps': 29.849096235697306, 'frame_indices': [0, 11, 22, 33, 44, 55, 66, 77, 88, 99, 110, 121, 132, 143, 154, 165, 176, 187, 198, 209, 220, 231, 242, 253, 264, 275, 286, 297, 308, 319, 330, 341]}, {'video_id': 'S2_FALL_21', 'sample_index': 75, 'frame_count': 283, 'decoded_frame_count': 283, 'fps': 29.82180680105799, 'frame_indices': [0, 9, 18, 27, 36, 45, 55, 64, 73, 82, 91, 100, 109, 118, 127, 136, 146, 155, 164, 173, 182, 191, 200, 209, 218, 227, 237, 246, 255, 264, 273, 282]}, {'video_id': 'S2_FALL_22', 'sample_index': 76, 'frame_count': 155, 'decoded_frame_count': 155, 'fps': 29.80005126890541, 'frame_indices': [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 79, 84, 89, 94, 99, 104, 109, 114, 119, 124, 129, 134, 139, 144, 149, 154]}, {'video_id': 'S2_FALL_23', 'sample_index': 77, 'frame_count': 225, 'decoded_frame_count': 225, 'fps': 29.849426227695844, 'frame_indices': [0, 7, 14, 22, 29, 36, 43, 51, 58, 65, 72, 79, 87, 94, 101, 108, 116, 123, 130, 137, 145, 152, 159, 166, 173, 181, 188, 195, 202, 210, 217, 224]}, {'video_id': 'S2_FALL_24', 'sample_index': 78, 'frame_count': 213, 'decoded_frame_count': 213, 'fps': 29.843218023622377, 'frame_indices': [0, 7, 14, 21, 27, 34, 41, 48, 55, 62, 68, 75, 82, 89, 96, 103, 109, 116, 123, 130, 137, 144, 150, 157, 164, 171, 178, 185, 191, 198, 205, 212]}, {'video_id': 'S2_FALL_25', 'sample_index': 79, 'frame_count': 218, 'decoded_frame_count': 218, 'fps': 29.871606900615244, 'frame_indices': [0, 7, 14, 21, 28, 35, 42, 49, 56, 63, 70, 77, 84, 91, 98, 105, 112, 119, 126, 133, 140, 147, 154, 161, 168, 175, 182, 189, 196, 203, 210, 217]}, {'video_id': 'S3_ADL_01', 'sample_index': 80, 'frame_count': 261, 'decoded_frame_count': 261, 'fps': 29.816644770663164, 'frame_indices': [0, 8, 17, 25, 34, 42, 50, 59, 67, 75, 84, 92, 101, 109, 117, 126, 134, 143, 151, 159, 168, 176, 185, 193, 201, 210, 218, 226, 235, 243, 252, 260]}, {'video_id': 'S3_ADL_02', 'sample_index': 81, 'frame_count': 273, 'decoded_frame_count': 273, 'fps': 29.8767719953598, 'frame_indices': [0, 9, 18, 26, 35, 44, 53, 61, 70, 79, 88, 97, 105, 114, 123, 132, 140, 149, 158, 167, 175, 184, 193, 202, 211, 219, 228, 237, 246, 254, 263, 272]}, {'video_id': 'S3_ADL_03', 'sample_index': 82, 'frame_count': 296, 'decoded_frame_count': 296, 'fps': 29.833396606115173, 'frame_indices': [0, 10, 19, 29, 38, 48, 57, 67, 76, 86, 95, 105, 114, 124, 133, 143, 152, 162, 171, 181, 190, 200, 209, 219, 228, 238, 247, 257, 266, 276, 285, 295]}, {'video_id': 'S3_ADL_04', 'sample_index': 83, 'frame_count': 300, 'decoded_frame_count': 300, 'fps': 29.852231454301208, 'frame_indices': [0, 10, 19, 29, 39, 48, 58, 68, 77, 87, 96, 106, 116, 125, 135, 145, 154, 164, 174, 183, 193, 203, 212, 222, 231, 241, 251, 260, 270, 280, 289, 299]}, {'video_id': 'S3_ADL_05', 'sample_index': 84, 'frame_count': 154, 'decoded_frame_count': 154, 'fps': 29.791267676474572, 'frame_indices': [0, 5, 10, 15, 20, 25, 30, 35, 39, 44, 49, 54, 59, 64, 69, 74, 79, 84, 89, 94, 99, 104, 109, 114, 118, 123, 128, 133, 138, 143, 148, 153]}, {'video_id': 'S3_ADL_06', 'sample_index': 85, 'frame_count': 267, 'decoded_frame_count': 267, 'fps': 29.847742974042525, 'frame_indices': [0, 9, 17, 26, 34, 43, 51, 60, 69, 77, 86, 94, 103, 112, 120, 129, 137, 146, 154, 163, 172, 180, 189, 197, 206, 215, 223, 232, 240, 249, 257, 266]}, {'video_id': 'S3_ADL_07', 'sample_index': 86, 'frame_count': 100, 'decoded_frame_count': 100, 'fps': 29.890898221491558, 'frame_indices': [0, 3, 6, 10, 13, 16, 19, 22, 26, 29, 32, 35, 38, 42, 45, 48, 51, 54, 57, 61, 64, 67, 70, 73, 77, 80, 83, 86, 89, 93, 96, 99]}, {'video_id': 'S3_ADL_08', 'sample_index': 87, 'frame_count': 146, 'decoded_frame_count': 146, 'fps': 29.819246349184738, 'frame_indices': [0, 5, 9, 14, 19, 23, 28, 33, 37, 42, 47, 51, 56, 61, 65, 70, 75, 80, 84, 89, 94, 98, 103, 108, 112, 117, 122, 126, 131, 136, 140, 145]}, {'video_id': 'S3_ADL_09', 'sample_index': 88, 'frame_count': 232, 'decoded_frame_count': 232, 'fps': 29.835518842244696, 'frame_indices': [0, 7, 15, 22, 30, 37, 45, 52, 60, 67, 75, 82, 89, 97, 104, 112, 119, 127, 134, 142, 149, 156, 164, 171, 179, 186, 194, 201, 209, 216, 224, 231]}, {'video_id': 'S3_ADL_10', 'sample_index': 89, 'frame_count': 371, 'decoded_frame_count': 371, 'fps': 29.838582117761867, 'frame_indices': [0, 12, 24, 36, 48, 60, 72, 84, 95, 107, 119, 131, 143, 155, 167, 179, 191, 203, 215, 227, 239, 251, 263, 275, 286, 298, 310, 322, 334, 346, 358, 370]}, {'video_id': 'S3_ADL_11', 'sample_index': 90, 'frame_count': 361, 'decoded_frame_count': 361, 'fps': 29.880395646235982, 'frame_indices': [0, 12, 23, 35, 46, 58, 70, 81, 93, 105, 116, 128, 139, 151, 163, 174, 186, 197, 209, 221, 232, 244, 255, 267, 279, 290, 302, 314, 325, 337, 348, 360]}, {'video_id': 'S3_ADL_12', 'sample_index': 91, 'frame_count': 160, 'decoded_frame_count': 160, 'fps': 14.990630855715178, 'frame_indices': [0, 5, 10, 15, 21, 26, 31, 36, 41, 46, 51, 56, 62, 67, 72, 77, 82, 87, 92, 97, 103, 108, 113, 118, 123, 128, 133, 138, 144, 149, 154, 159]}, {'video_id': 'S3_ADL_13', 'sample_index': 92, 'frame_count': 130, 'decoded_frame_count': 130, 'fps': 15.043684545507146, 'frame_indices': [0, 4, 8, 12, 17, 21, 25, 29, 33, 37, 42, 46, 50, 54, 58, 62, 67, 71, 75, 79, 83, 87, 92, 96, 100, 104, 108, 112, 117, 121, 125, 129]}, {'video_id': 'S3_ADL_14', 'sample_index': 93, 'frame_count': 133, 'decoded_frame_count': 133, 'fps': 15.001748324052803, 'frame_indices': [0, 4, 9, 13, 17, 21, 26, 30, 34, 38, 43, 47, 51, 55, 60, 64, 68, 72, 77, 81, 85, 89, 94, 98, 102, 106, 111, 115, 119, 123, 128, 132]}, {'video_id': 'S3_ADL_15', 'sample_index': 94, 'frame_count': 115, 'decoded_frame_count': 115, 'fps': 15.00280487221524, 'frame_indices': [0, 4, 7, 11, 15, 18, 22, 26, 29, 33, 37, 40, 44, 48, 51, 55, 59, 63, 66, 70, 74, 77, 81, 85, 88, 92, 96, 99, 103, 107, 110, 114]}, {'video_id': 'S3_ADL_16', 'sample_index': 95, 'frame_count': 306, 'decoded_frame_count': 306, 'fps': 29.83250898549971, 'frame_indices': [0, 10, 20, 30, 39, 49, 59, 69, 79, 89, 98, 108, 118, 128, 138, 148, 157, 167, 177, 187, 197, 207, 216, 226, 236, 246, 256, 266, 275, 285, 295, 305]}, {'video_id': 'S3_ADL_17', 'sample_index': 96, 'frame_count': 182, 'decoded_frame_count': 182, 'fps': 15.00482297881462, 'frame_indices': [0, 6, 12, 18, 23, 29, 35, 41, 47, 53, 58, 64, 70, 76, 82, 88, 93, 99, 105, 111, 117, 123, 128, 134, 140, 146, 152, 158, 163, 169, 175, 181]}, {'video_id': 'S3_ADL_18', 'sample_index': 97, 'frame_count': 167, 'decoded_frame_count': 167, 'fps': 15.0160050832624, 'frame_indices': [0, 5, 11, 16, 21, 27, 32, 37, 43, 48, 54, 59, 64, 70, 75, 80, 86, 91, 96, 102, 107, 112, 118, 123, 129, 134, 139, 145, 150, 155, 161, 166]}, {'video_id': 'S3_ADL_19', 'sample_index': 98, 'frame_count': 98, 'decoded_frame_count': 98, 'fps': 15.046136367125728, 'frame_indices': [0, 3, 6, 9, 13, 16, 19, 22, 25, 28, 31, 34, 38, 41, 44, 47, 50, 53, 56, 59, 63, 66, 69, 72, 75, 78, 81, 84, 88, 91, 94, 97]}, {'video_id': 'S3_ADL_20', 'sample_index': 99, 'frame_count': 141, 'decoded_frame_count': 141, 'fps': 15.010699115326881, 'frame_indices': [0, 5, 9, 14, 18, 23, 27, 32, 36, 41, 45, 50, 54, 59, 63, 68, 72, 77, 81, 86, 90, 95, 99, 104, 108, 113, 117, 122, 126, 131, 135, 140]}, {'video_id': 'S3_ADL_21', 'sample_index': 100, 'frame_count': 305, 'decoded_frame_count': 305, 'fps': 29.82826741775222, 'frame_indices': [0, 10, 20, 29, 39, 49, 59, 69, 78, 88, 98, 108, 118, 127, 137, 147, 157, 167, 177, 186, 196, 206, 216, 226, 235, 245, 255, 265, 275, 284, 294, 304]}, {'video_id': 'S3_ADL_22', 'sample_index': 101, 'frame_count': 283, 'decoded_frame_count': 283, 'fps': 29.822121058843365, 'frame_indices': [0, 9, 18, 27, 36, 45, 55, 64, 73, 82, 91, 100, 109, 118, 127, 136, 146, 155, 164, 173, 182, 191, 200, 209, 218, 227, 237, 246, 255, 264, 273, 282]}, {'video_id': 'S3_FALL_01', 'sample_index': 102, 'frame_count': 114, 'decoded_frame_count': 114, 'fps': 29.925187032418954, 'frame_indices': [0, 4, 7, 11, 15, 18, 22, 26, 29, 33, 36, 40, 44, 47, 51, 55, 58, 62, 66, 69, 73, 77, 80, 84, 87, 91, 95, 98, 102, 106, 109, 113]}, {'video_id': 'S3_FALL_02', 'sample_index': 103, 'frame_count': 196, 'decoded_frame_count': 196, 'fps': 29.870307999451363, 'frame_indices': [0, 6, 13, 19, 25, 31, 38, 44, 50, 57, 63, 69, 75, 82, 88, 94, 101, 107, 113, 120, 126, 132, 138, 145, 151, 157, 164, 170, 176, 182, 189, 195]}, {'video_id': 'S3_FALL_03', 'sample_index': 104, 'frame_count': 167, 'decoded_frame_count': 167, 'fps': 29.812733190914553, 'frame_indices': [0, 5, 11, 16, 21, 27, 32, 37, 43, 48, 54, 59, 64, 70, 75, 80, 86, 91, 96, 102, 107, 112, 118, 123, 129, 134, 139, 145, 150, 155, 161, 166]}, {'video_id': 'S3_FALL_04', 'sample_index': 105, 'frame_count': 163, 'decoded_frame_count': 163, 'fps': 29.866972869306036, 'frame_indices': [0, 5, 10, 16, 21, 26, 31, 37, 42, 47, 52, 57, 63, 68, 73, 78, 84, 89, 94, 99, 105, 110, 115, 120, 125, 131, 136, 141, 146, 152, 157, 162]}, {'video_id': 'S3_FALL_05', 'sample_index': 106, 'frame_count': 300, 'decoded_frame_count': 300, 'fps': 29.852627528766323, 'frame_indices': [0, 10, 19, 29, 39, 48, 58, 68, 77, 87, 96, 106, 116, 125, 135, 145, 154, 164, 174, 183, 193, 203, 212, 222, 231, 241, 251, 260, 270, 280, 289, 299]}, {'video_id': 'S3_FALL_06', 'sample_index': 107, 'frame_count': 166, 'decoded_frame_count': 166, 'fps': 29.89034205835219, 'frame_indices': [0, 5, 11, 16, 21, 27, 32, 37, 43, 48, 53, 59, 64, 69, 75, 80, 85, 90, 96, 101, 106, 112, 117, 122, 128, 133, 138, 144, 149, 154, 160, 165]}, {'video_id': 'S3_FALL_07', 'sample_index': 108, 'frame_count': 269, 'decoded_frame_count': 269, 'fps': 29.804773195647837, 'frame_indices': [0, 9, 17, 26, 35, 43, 52, 61, 69, 78, 86, 95, 104, 112, 121, 130, 138, 147, 156, 164, 173, 182, 190, 199, 207, 216, 225, 233, 242, 251, 259, 268]}, {'video_id': 'S3_FALL_08', 'sample_index': 109, 'frame_count': 186, 'decoded_frame_count': 186, 'fps': 29.35003865999716, 'frame_indices': [0, 6, 12, 18, 24, 30, 36, 42, 48, 54, 60, 66, 72, 78, 84, 90, 95, 101, 107, 113, 119, 125, 131, 137, 143, 149, 155, 161, 167, 173, 179, 185]}, {'video_id': 'S3_FALL_09', 'sample_index': 110, 'frame_count': 186, 'decoded_frame_count': 186, 'fps': 29.88319846621003, 'frame_indices': [0, 6, 12, 18, 24, 30, 36, 42, 48, 54, 60, 66, 72, 78, 84, 90, 95, 101, 107, 113, 119, 125, 131, 137, 143, 149, 155, 161, 167, 173, 179, 185]}, {'video_id': 'S3_FALL_10', 'sample_index': 111, 'frame_count': 131, 'decoded_frame_count': 131, 'fps': 29.87071226067327, 'frame_indices': [0, 4, 8, 13, 17, 21, 25, 29, 34, 38, 42, 46, 50, 55, 59, 63, 67, 71, 75, 80, 84, 88, 92, 96, 101, 105, 109, 113, 117, 122, 126, 130]}, {'video_id': 'S3_FALL_11', 'sample_index': 112, 'frame_count': 267, 'decoded_frame_count': 267, 'fps': 29.847409311944553, 'frame_indices': [0, 9, 17, 26, 34, 43, 51, 60, 69, 77, 86, 94, 103, 112, 120, 129, 137, 146, 154, 163, 172, 180, 189, 197, 206, 215, 223, 232, 240, 249, 257, 266]}, {'video_id': 'S3_FALL_12', 'sample_index': 113, 'frame_count': 127, 'decoded_frame_count': 127, 'fps': 29.839057054469986, 'frame_indices': [0, 4, 8, 12, 16, 20, 24, 28, 33, 37, 41, 45, 49, 53, 57, 61, 65, 69, 73, 77, 81, 85, 89, 93, 98, 102, 106, 110, 114, 118, 122, 126]}, {'video_id': 'S3_FALL_13', 'sample_index': 114, 'frame_count': 121, 'decoded_frame_count': 121, 'fps': 29.889581463519065, 'frame_indices': [0, 4, 8, 12, 15, 19, 23, 27, 31, 35, 39, 43, 46, 50, 54, 58, 62, 66, 70, 74, 77, 81, 85, 89, 93, 97, 101, 105, 108, 112, 116, 120]}, {'video_id': 'S3_FALL_14', 'sample_index': 115, 'frame_count': 367, 'decoded_frame_count': 367, 'fps': 29.8238200494084, 'frame_indices': [0, 12, 24, 35, 47, 59, 71, 83, 94, 106, 118, 130, 142, 153, 165, 177, 189, 201, 213, 224, 236, 248, 260, 272, 283, 295, 307, 319, 331, 342, 354, 366]}, {'video_id': 'S3_FALL_15', 'sample_index': 116, 'frame_count': 85, 'decoded_frame_count': 85, 'fps': 15.04522417384019, 'frame_indices': [0, 3, 5, 8, 11, 14, 16, 19, 22, 24, 27, 30, 33, 35, 38, 41, 43, 46, 49, 51, 54, 57, 60, 62, 65, 68, 70, 73, 76, 79, 81, 84]}, {'video_id': 'S3_FALL_16', 'sample_index': 117, 'frame_count': 91, 'decoded_frame_count': 91, 'fps': 14.96636679111228, 'frame_indices': [0, 3, 6, 9, 12, 15, 17, 20, 23, 26, 29, 32, 35, 38, 41, 44, 46, 49, 52, 55, 58, 61, 64, 67, 70, 73, 75, 78, 81, 84, 87, 90]}, {'video_id': 'S3_FALL_17', 'sample_index': 118, 'frame_count': 128, 'decoded_frame_count': 128, 'fps': 15.035179971887345, 'frame_indices': [0, 4, 8, 12, 16, 20, 25, 29, 33, 37, 41, 45, 49, 53, 57, 61, 66, 70, 74, 78, 82, 86, 90, 94, 98, 102, 107, 111, 115, 119, 123, 127]}, {'video_id': 'S3_FALL_18', 'sample_index': 119, 'frame_count': 178, 'decoded_frame_count': 178, 'fps': 14.991241072631722, 'frame_indices': [0, 6, 11, 17, 23, 29, 34, 40, 46, 51, 57, 63, 69, 74, 80, 86, 91, 97, 103, 108, 114, 120, 126, 131, 137, 143, 148, 154, 160, 166, 171, 177]}, {'video_id': 'S3_FALL_19', 'sample_index': 120, 'frame_count': 86, 'decoded_frame_count': 86, 'fps': 15.051630593314275, 'frame_indices': [0, 3, 5, 8, 11, 14, 16, 19, 22, 25, 27, 30, 33, 36, 38, 41, 44, 47, 49, 52, 55, 58, 60, 63, 66, 69, 71, 74, 77, 80, 82, 85]}, {'video_id': 'S3_FALL_20', 'sample_index': 121, 'frame_count': 94, 'decoded_frame_count': 94, 'fps': 15.060723555612523, 'frame_indices': [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33, 36, 39, 42, 45, 48, 51, 54, 57, 60, 63, 66, 69, 72, 75, 78, 81, 84, 87, 90, 93]}, {'video_id': 'S3_FALL_21', 'sample_index': 122, 'frame_count': 274, 'decoded_frame_count': 274, 'fps': 29.881238435560856, 'frame_indices': [0, 9, 18, 26, 35, 44, 53, 62, 70, 79, 88, 97, 106, 114, 123, 132, 141, 150, 159, 167, 176, 185, 194, 203, 211, 220, 229, 238, 247, 255, 264, 273]}, {'video_id': 'S4_ADL_01', 'sample_index': 123, 'frame_count': 370, 'decoded_frame_count': 370, 'fps': 29.835341171158095, 'frame_indices': [0, 12, 24, 36, 48, 60, 71, 83, 95, 107, 119, 131, 143, 155, 167, 179, 190, 202, 214, 226, 238, 250, 262, 274, 286, 298, 309, 321, 333, 345, 357, 369]}, {'video_id': 'S4_ADL_02', 'sample_index': 124, 'frame_count': 315, 'decoded_frame_count': 315, 'fps': 29.82540303745692, 'frame_indices': [0, 10, 20, 30, 41, 51, 61, 71, 81, 91, 101, 111, 122, 132, 142, 152, 162, 172, 182, 192, 203, 213, 223, 233, 243, 253, 263, 273, 284, 294, 304, 314]}, {'video_id': 'S4_ADL_03', 'sample_index': 125, 'frame_count': 267, 'decoded_frame_count': 267, 'fps': 29.84674201012777, 'frame_indices': [0, 9, 17, 26, 34, 43, 51, 60, 69, 77, 86, 94, 103, 112, 120, 129, 137, 146, 154, 163, 172, 180, 189, 197, 206, 215, 223, 232, 240, 249, 257, 266]}, {'video_id': 'S4_ADL_04', 'sample_index': 126, 'frame_count': 388, 'decoded_frame_count': 388, 'fps': 29.82443553924834, 'frame_indices': [0, 12, 25, 37, 50, 62, 75, 87, 100, 112, 125, 137, 150, 162, 175, 187, 200, 212, 225, 237, 250, 262, 275, 287, 300, 312, 325, 337, 350, 362, 375, 387]}, {'video_id': 'S4_ADL_05', 'sample_index': 127, 'frame_count': 220, 'decoded_frame_count': 220, 'fps': 29.88453701607426, 'frame_indices': [0, 7, 14, 21, 28, 35, 42, 49, 57, 64, 71, 78, 85, 92, 99, 106, 113, 120, 127, 134, 141, 148, 155, 162, 170, 177, 184, 191, 198, 205, 212, 219]}, {'video_id': 'S4_ADL_06', 'sample_index': 128, 'frame_count': 196, 'decoded_frame_count': 196, 'fps': 29.79810265142301, 'frame_indices': [0, 6, 13, 19, 25, 31, 38, 44, 50, 57, 63, 69, 75, 82, 88, 94, 101, 107, 113, 120, 126, 132, 138, 145, 151, 157, 164, 170, 176, 182, 189, 195]}, {'video_id': 'S4_ADL_07', 'sample_index': 129, 'frame_count': 149, 'decoded_frame_count': 149, 'fps': 29.83938799214964, 'frame_indices': [0, 5, 10, 14, 19, 24, 29, 33, 38, 43, 48, 53, 57, 62, 67, 72, 76, 81, 86, 91, 95, 100, 105, 110, 115, 119, 124, 129, 134, 138, 143, 148]}, {'video_id': 'S4_ADL_08', 'sample_index': 130, 'frame_count': 368, 'decoded_frame_count': 368, 'fps': 29.82776089159068, 'frame_indices': [0, 12, 24, 36, 47, 59, 71, 83, 95, 107, 118, 130, 142, 154, 166, 178, 189, 201, 213, 225, 237, 249, 260, 272, 284, 296, 308, 320, 331, 343, 355, 367]}, {'video_id': 'S4_ADL_09', 'sample_index': 131, 'frame_count': 362, 'decoded_frame_count': 362, 'fps': 29.84467577579668, 'frame_indices': [0, 12, 23, 35, 47, 58, 70, 82, 93, 105, 116, 128, 140, 151, 163, 175, 186, 198, 210, 221, 233, 245, 256, 268, 279, 291, 303, 314, 326, 338, 349, 361]}, {'video_id': 'S4_ADL_10', 'sample_index': 132, 'frame_count': 382, 'decoded_frame_count': 382, 'fps': 29.840952410711495, 'frame_indices': [0, 12, 25, 37, 49, 61, 74, 86, 98, 111, 123, 135, 147, 160, 172, 184, 197, 209, 221, 234, 246, 258, 270, 283, 295, 307, 320, 332, 344, 356, 369, 381]}, {'video_id': 'S4_ADL_11', 'sample_index': 133, 'frame_count': 320, 'decoded_frame_count': 320, 'fps': 29.846662770019027, 'frame_indices': [0, 10, 21, 31, 41, 51, 62, 72, 82, 93, 103, 113, 123, 134, 144, 154, 165, 175, 185, 196, 206, 216, 226, 237, 247, 257, 268, 278, 288, 298, 309, 319]}, {'video_id': 'S4_ADL_12', 'sample_index': 134, 'frame_count': 257, 'decoded_frame_count': 257, 'fps': 29.85143972216092, 'frame_indices': [0, 8, 17, 25, 33, 41, 50, 58, 66, 74, 83, 91, 99, 107, 116, 124, 132, 140, 149, 157, 165, 173, 182, 190, 198, 206, 215, 223, 231, 239, 248, 256]}, {'video_id': 'S4_ADL_13', 'sample_index': 135, 'frame_count': 164, 'decoded_frame_count': 164, 'fps': 29.875034914929017, 'frame_indices': [0, 5, 11, 16, 21, 26, 32, 37, 42, 47, 53, 58, 63, 68, 74, 79, 84, 89, 95, 100, 105, 110, 116, 121, 126, 131, 137, 142, 147, 152, 158, 163]}, {'video_id': 'S4_ADL_14', 'sample_index': 136, 'frame_count': 248, 'decoded_frame_count': 248, 'fps': 29.859132319300077, 'frame_indices': [0, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 127, 135, 143, 151, 159, 167, 175, 183, 191, 199, 207, 215, 223, 231, 239, 247]}, {'video_id': 'S4_ADL_15', 'sample_index': 137, 'frame_count': 159, 'decoded_frame_count': 159, 'fps': 29.83356995878339, 'frame_indices': [0, 5, 10, 15, 20, 25, 31, 36, 41, 46, 51, 56, 61, 66, 71, 76, 82, 87, 92, 97, 102, 107, 112, 117, 122, 127, 133, 138, 143, 148, 153, 158]}, {'video_id': 'S4_ADL_16', 'sample_index': 138, 'frame_count': 213, 'decoded_frame_count': 213, 'fps': 29.842103050993103, 'frame_indices': [0, 7, 14, 21, 27, 34, 41, 48, 55, 62, 68, 75, 82, 89, 96, 103, 109, 116, 123, 130, 137, 144, 150, 157, 164, 171, 178, 185, 191, 198, 205, 212]}, {'video_id': 'S4_ADL_17', 'sample_index': 139, 'frame_count': 186, 'decoded_frame_count': 186, 'fps': 29.877758204336022, 'frame_indices': [0, 6, 12, 18, 24, 30, 36, 42, 48, 54, 60, 66, 72, 78, 84, 90, 95, 101, 107, 113, 119, 125, 131, 137, 143, 149, 155, 161, 167, 173, 179, 185]}, {'video_id': 'S4_ADL_18', 'sample_index': 140, 'frame_count': 421, 'decoded_frame_count': 421, 'fps': 29.863592190540594, 'frame_indices': [0, 14, 27, 41, 54, 68, 81, 95, 108, 122, 135, 149, 163, 176, 190, 203, 217, 230, 244, 257, 271, 285, 298, 312, 325, 339, 352, 366, 379, 393, 406, 420]}, {'video_id': 'S4_ADL_19', 'sample_index': 141, 'frame_count': 296, 'decoded_frame_count': 296, 'fps': 29.882891371651635, 'frame_indices': [0, 10, 19, 29, 38, 48, 57, 67, 76, 86, 95, 105, 114, 124, 133, 143, 152, 162, 171, 181, 190, 200, 209, 219, 228, 238, 247, 257, 266, 276, 285, 295]}, {'video_id': 'S4_ADL_20', 'sample_index': 142, 'frame_count': 523, 'decoded_frame_count': 523, 'fps': 29.84921296243451, 'frame_indices': [0, 17, 34, 51, 67, 84, 101, 118, 135, 152, 168, 185, 202, 219, 236, 253, 269, 286, 303, 320, 337, 354, 370, 387, 404, 421, 438, 455, 471, 488, 505, 522]}, {'video_id': 'S4_FALL_01', 'sample_index': 143, 'frame_count': 139, 'decoded_frame_count': 139, 'fps': 29.844337090713903, 'frame_indices': [0, 4, 9, 13, 18, 22, 27, 31, 36, 40, 45, 49, 53, 58, 62, 67, 71, 76, 80, 85, 89, 93, 98, 102, 107, 111, 116, 120, 125, 129, 134, 138]}, {'video_id': 'S4_FALL_02', 'sample_index': 144, 'frame_count': 89, 'decoded_frame_count': 89, 'fps': 29.892186607852576, 'frame_indices': [0, 3, 6, 9, 11, 14, 17, 20, 23, 26, 28, 31, 34, 37, 40, 43, 45, 48, 51, 54, 57, 60, 62, 65, 68, 71, 74, 77, 79, 82, 85, 88]}, {'video_id': 'S4_FALL_03', 'sample_index': 145, 'frame_count': 200, 'decoded_frame_count': 200, 'fps': 29.82611375679787, 'frame_indices': [0, 6, 13, 19, 26, 32, 39, 45, 51, 58, 64, 71, 77, 83, 90, 96, 103, 109, 116, 122, 128, 135, 141, 148, 154, 160, 167, 173, 180, 186, 193, 199]}, {'video_id': 'S4_FALL_04', 'sample_index': 146, 'frame_count': 141, 'decoded_frame_count': 141, 'fps': 29.862969212196518, 'frame_indices': [0, 5, 9, 14, 18, 23, 27, 32, 36, 41, 45, 50, 54, 59, 63, 68, 72, 77, 81, 86, 90, 95, 99, 104, 108, 113, 117, 122, 126, 131, 135, 140]}, {'video_id': 'S4_FALL_05', 'sample_index': 147, 'frame_count': 221, 'decoded_frame_count': 221, 'fps': 29.826037275798623, 'frame_indices': [0, 7, 14, 21, 28, 35, 43, 50, 57, 64, 71, 78, 85, 92, 99, 106, 114, 121, 128, 135, 142, 149, 156, 163, 170, 177, 185, 192, 199, 206, 213, 220]}, {'video_id': 'S4_FALL_06', 'sample_index': 148, 'frame_count': 183, 'decoded_frame_count': 183, 'fps': 29.85496685500492, 'frame_indices': [0, 6, 12, 18, 23, 29, 35, 41, 47, 53, 59, 65, 70, 76, 82, 88, 94, 100, 106, 112, 117, 123, 129, 135, 141, 147, 153, 159, 164, 170, 176, 182]}, {'video_id': 'S4_FALL_07', 'sample_index': 149, 'frame_count': 188, 'decoded_frame_count': 188, 'fps': 29.817447436175332, 'frame_indices': [0, 6, 12, 18, 24, 30, 36, 42, 48, 54, 60, 66, 72, 78, 84, 90, 97, 103, 109, 115, 121, 127, 133, 139, 145, 151, 157, 163, 169, 175, 181, 187]}, {'video_id': 'S4_FALL_08', 'sample_index': 150, 'frame_count': 187, 'decoded_frame_count': 187, 'fps': 29.883979842962617, 'frame_indices': [0, 6, 12, 18, 24, 30, 36, 42, 48, 54, 60, 66, 72, 78, 84, 90, 96, 102, 108, 114, 120, 126, 132, 138, 144, 150, 156, 162, 168, 174, 180, 186]}, {'video_id': 'S4_FALL_09', 'sample_index': 151, 'frame_count': 181, 'decoded_frame_count': 181, 'fps': 29.839755566790494, 'frame_indices': [0, 6, 12, 17, 23, 29, 35, 41, 46, 52, 58, 64, 70, 75, 81, 87, 93, 99, 105, 110, 116, 122, 128, 134, 139, 145, 151, 157, 163, 168, 174, 180]}, {'video_id': 'S4_FALL_10', 'sample_index': 152, 'frame_count': 151, 'decoded_frame_count': 151, 'fps': 29.856254984280977, 'frame_indices': [0, 5, 10, 15, 19, 24, 29, 34, 39, 44, 48, 53, 58, 63, 68, 73, 77, 82, 87, 92, 97, 102, 106, 111, 116, 121, 126, 131, 135, 140, 145, 150]}, {'video_id': 'S4_FALL_11', 'sample_index': 153, 'frame_count': 136, 'decoded_frame_count': 136, 'fps': 29.814971792698255, 'frame_indices': [0, 4, 9, 13, 17, 22, 26, 30, 35, 39, 44, 48, 52, 57, 61, 65, 70, 74, 78, 83, 87, 91, 96, 100, 105, 109, 113, 118, 122, 126, 131, 135]}, {'video_id': 'S4_FALL_12', 'sample_index': 154, 'frame_count': 138, 'decoded_frame_count': 138, 'fps': 29.83568751801672, 'frame_indices': [0, 4, 9, 13, 18, 22, 27, 31, 35, 40, 44, 49, 53, 57, 62, 66, 71, 75, 80, 84, 88, 93, 97, 102, 106, 110, 115, 119, 124, 128, 133, 137]}, {'video_id': 'S4_FALL_13', 'sample_index': 155, 'frame_count': 149, 'decoded_frame_count': 149, 'fps': 29.839587185666318, 'frame_indices': [0, 5, 10, 14, 19, 24, 29, 33, 38, 43, 48, 53, 57, 62, 67, 72, 76, 81, 86, 91, 95, 100, 105, 110, 115, 119, 124, 129, 134, 138, 143, 148]}, {'video_id': 'S4_FALL_14', 'sample_index': 156, 'frame_count': 157, 'decoded_frame_count': 157, 'fps': 29.907230438067906, 'frame_indices': [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 81, 86, 91, 96, 101, 106, 111, 116, 121, 126, 131, 136, 141, 146, 151, 156]}, {'video_id': 'S4_FALL_15', 'sample_index': 157, 'frame_count': 145, 'decoded_frame_count': 145, 'fps': 29.90019521046988, 'frame_indices': [0, 5, 9, 14, 19, 23, 28, 33, 37, 42, 46, 51, 56, 60, 65, 70, 74, 79, 84, 88, 93, 98, 102, 107, 111, 116, 121, 125, 130, 135, 139, 144]}, {'video_id': 'S4_FALL_16', 'sample_index': 158, 'frame_count': 212, 'decoded_frame_count': 212, 'fps': 29.83548264522515, 'frame_indices': [0, 7, 14, 20, 27, 34, 41, 48, 54, 61, 68, 75, 82, 88, 95, 102, 109, 116, 123, 129, 136, 143, 150, 157, 163, 170, 177, 184, 191, 197, 204, 211]}, {'video_id': 'S4_FALL_17', 'sample_index': 159, 'frame_count': 207, 'decoded_frame_count': 207, 'fps': 29.8725725528302, 'frame_indices': [0, 7, 13, 20, 27, 33, 40, 47, 53, 60, 66, 73, 80, 86, 93, 100, 106, 113, 120, 126, 133, 140, 146, 153, 159, 166, 173, 179, 186, 193, 199, 206]}]}
actual_hash = hashlib.sha256(json.dumps(SOURCE_FILES, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()
if actual_hash != CODE_HASH:
    raise ValueError("Mã nhúng đã đổi; không chạy chung với cấu hình cũ.")
RUNTIME_ROOT = OUTPUT_ROOT / "_08_runtime"
for relative_path, source in SOURCE_FILES.items():
    destination = RUNTIME_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding="utf-8")
for package in ["research07", "research08"]:
    (RUNTIME_ROOT / package / "__init__.py").write_text("", encoding="utf-8")
for module_name in list(sys.modules):
    if module_name in {"research07", "research08"} or module_name.startswith(("research07.", "research08.")):
        del sys.modules[module_name]
sys.path.insert(0, str(RUNTIME_ROOT))
from research07.features import load_inputs
from research07.training import write_json
from research08.features import attach_nominal_timing
from research08.validate_pipeline import check_pipeline
from research08.training import default_config, run_experiment
from research08.reporting import export_figures
print("Mã độc lập đã nạp | SHA256:", CODE_HASH[:16])
print("Bằng chứng frame:", TIMING_EVIDENCE["samples"], "video |", TIMING_EVIDENCE["timing_type"])

## Cell 3 — Đọc bốn input và kiểm tra căn chỉnh
Tìm file đệ quy trong /kaggle/input nên không phụ thuộc tên tài khoản/dataset. Chấp nhận các bản tải cũ đổi đuôi NPZ thành ZIP và CSV thành TXT. Không hỗ trợ ZIP thật chứa nhiều file chưa giải nén. Bản trùng nội dung được chấp nhận; bản cùng tên nhưng khác nội dung gây lỗi rõ ràng.

Tệp chia tập cũ được kiểm tra làm bằng chứng căn chỉnh. Thí nghiệm vẫn tự tạo đầy đủ nested LOSO: một người test, ba lượt inner luân phiên người validation trong ba người còn lại; không chỉ dùng một split train/val đã lưu.

In [ ]:
INPUT_ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd() / "Kaggle_kiem_tra_2026-09-25"
data = load_inputs(INPUT_ROOT)
diagnostics = attach_nominal_timing(data, TIMING_EVIDENCE)
print("Mẫu:", len(data["labels"]), "| ADL:", int((data["labels"] == 0).sum()), "| Fall:", int((data["labels"] == 1).sum()))
print("Chủ thể:", np.unique(data["subjects"]).tolist())
print("Mốc thiếu phát hiện trực tiếp:", int((~data["masks"]).sum()))
print("Đã căn chỉnh FPS, số frame giải mã và 32 mốc cho đủ 160 video.")
print("Sai khác tương đối dt lớn nhất:", f"{diagnostics['max_relative_dt_difference'].max():.2%}")
display(diagnostics.head())

## Cell 4 — Kiểm tra kỹ thuật trước khi train
Kiểm tra đối chứng trùng đặc trưng 07, sáu kênh ngoài vận tốc không đổi, vận tốc tổng hợp biết trước, crop thời gian và augmentation dùng cùng chuỗi ngẫu nhiên. Kiểm tra này không đo hiệu quả nhận diện té ngã.

In [ ]:
pipeline_checks = check_pipeline(data)
print(json.dumps(pipeline_checks, ensure_ascii=False, indent=2))
print("Kiểm tra kỹ thuật đạt; CHƯA phải kết quả huấn luyện.")

## Cell 5 — Khóa cấu hình trước khi chạy
Mặc định full: seed 42/142/242, tối đa 50 epoch, patience 10, batch 16, AdamW lr 0,001, weight decay 0,0001 và dropout 0,3. Class weight chỉ tính từ train của từng lượt fit, Fall nhân thêm 2,25. Inner chọn epoch theo CE không trọng số, refit dùng trung vị ba best epoch.

Kết quả chính ở ngưỡng cố định 0,5. Các ngưỡng balanced/sensitivity chỉ chọn từ inner validation; sensitivity mục tiêu 0,95 và specificity tối thiểu 0,75 không phải lời hứa cho test. Không dùng outer score để chỉnh cấu hình.

Giữ RUN_MODE = full khi chạy Kaggle. Nếu cần tiếp tục phiên bản bị ngắt, gắn output phiên bản cũ và đặt RESUME_FROM thành đường dẫn thư mục 08_timing_controlled_ablation_full trong input. Bản mới kiểm tra code, dữ liệu, cấu hình và môi trường trước khi tiếp tục; khác nhau thì dừng. Không chọn lại hyperparameter dựa trên kết quả đã xem.

In [ ]:
RUN_MODE = "full"
RESUME_FROM = None
config = default_config(RUN_MODE)
RUN_DIR = OUTPUT_ROOT / ("08_timing_controlled_ablation_" + RUN_MODE)
if RESUME_FROM is not None:
    resume_path = Path(RESUME_FROM)
    if not (resume_path / "locked_protocol.json").is_file():
        raise FileNotFoundError("RESUME_FROM phải trỏ đến thư mục lượt chạy có locked_protocol.json")
    if RUN_DIR.exists():
        raise ValueError("Thư mục đích đã tồn tại; không trộn hai output khác nhau.")
    shutil.copytree(resume_path, RUN_DIR)
if RUN_MODE == "full" and not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Bật accelerator GPU trong Kaggle rồi Run All lại.")
print(json.dumps(config, ensure_ascii=False, indent=2))
print("Lưu kết quả:", RUN_DIR)
print("24 outer run / 96 lượt fit" if RUN_MODE == "full" else "SMOKE: 8 outer run / 32 lượt fit, không phải kết quả nghiên cứu")

## Cell 6 — Huấn luyện hai nhánh
Tiến độ được in sau mỗi inner fit và outer run. Mỗi inner hoàn thành được lưu để tiếp tục; mỗi outer lưu checkpoint, attention, loss và dự đoán. Không thay model web, không export model đã học toàn bộ 160 video. Chạy nhanh/dừng sớm không tự chứng minh mô hình tốt hoặc xấu.

In [ ]:
summary = run_experiment(data, config, RUN_DIR, CODE_HASH)
write_json(RUN_DIR / "pipeline_checks.json", pipeline_checks)
diagnostics.to_csv(RUN_DIR / "timing_diagnostics_160.csv", index=False)
for relative_path, source in SOURCE_FILES.items():
    destination = RUN_DIR / "source" / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding="utf-8")
print("Hoàn thành giao thức full." if RUN_MODE == "full" else "Chỉ hoàn thành smoke kiểm tra phần mềm.")

## Cell 7 — Đối chiếu kết quả, không tự chọn model thắng
Hiển thị đủ seed và cả FN/FP. Bảng paired_metrics tính nhánh mới trừ nhánh cũ: delta_FN âm nghĩa là ít bỏ sót hơn, delta_FP dương nghĩa là nhiều báo nhầm hơn. paired_predictions chỉ rõ ca được cứu và ca mới sai.

Độ lệch chuẩn qua seed phản ánh khởi tạo ngẫu nhiên trên cùng dữ liệu, không phải độ bất định dân số. Ba seed không biến 160 video thành 480 video độc lập. Ngưỡng theo inner có thể khác giữa hai nhánh; so sánh ngưỡng cố định 0,5 là phân tích chính để hạn chế nhiễu do chọn ngưỡng. Attention không phải chứng cứ nhân quả.

In [ ]:
pooled = pd.read_csv(RUN_DIR / "pooled_by_seed.csv")
paired = pd.read_csv(RUN_DIR / "paired_metrics.csv")
print("KẾT QUẢ CHÍNH: NGƯỠNG 0,5")
display(pooled.loc[pooled["policy"] == "fixed", ["variant", "seed", "accuracy", "sensitivity", "specificity", "f1", "fn", "fp"]])
display(paired.loc[paired["policy"] == "fixed", ["seed", "fn_legacy", "fn_interval", "delta_fn_interval_minus_legacy", "fp_legacy", "fp_interval", "delta_fp_interval_minus_legacy"]])
print("TẤT CẢ CHÍNH SÁCH NGƯỠNG")
display(pooled)
report_checks = export_figures(RUN_DIR, config)
write_json(RUN_DIR / "report_checks.json", report_checks)
print("Kiểm tra giới hạn epoch:", report_checks)
for figure_name in report_checks["figures"]:
    display(Image(filename=str(RUN_DIR / figure_name)))
if RUN_MODE == "smoke":
    print("SMOKE ONLY — không dùng bảng trên trong báo cáo kết quả mô hình.")

## Cell 8 — Đóng gói output để tải về
ZIP gồm bảng kết quả, dự đoán, checkpoint từng outer, loss, lựa chọn inner, timing, mã nguồn và cấu hình. Không chứa video hoặc toàn bộ pose/features đầu vào. Checkpoint outer không được đưa thẳng vào web như model final. Sau khi tải ZIP, cần phân tích cả cải thiện và ca mới sai trước khi quyết định bước tiếp theo.

Tên ZIP full: gmdcsa24_08_timing_controlled_ablation_full.zip. Không tải nhầm ZIP smoke.

In [ ]:
readme = """# 08 — Timing-controlled ablation
T0_legacy_dt and T1_interval_dt share ME-STGCN-TA, seeds, splits and augmentation.
Only two joint-velocity channels change their time denominator.
Nominal frame-index/FPS timing is not original presentation timing or physical m/s velocity.
Primary report: fixed threshold 0.5. Other thresholds use inner validation only.
Read completion.json before interpreting metrics. Smoke is not a scientific model result.
Full is development evaluation on previously inspected data, not new blind confirmation.
Use pooled_by_seed.csv, fold_metrics.csv, paired_metrics.csv and paired_predictions.csv together.
Do not choose the best seed. Do not pool seeds as independent videos.
Outer checkpoints are not all-subject deployment checkpoints. Model web 06 is unchanged.
"""
(RUN_DIR / "README.md").write_text(readme, encoding="utf-8")
completion = json.loads((RUN_DIR / "completion.json").read_text(encoding="utf-8"))
print(json.dumps(completion, ensure_ascii=False, indent=2))
archive_base = OUTPUT_ROOT / ("gmdcsa24_08_timing_controlled_ablation_" + RUN_MODE)
archive = shutil.make_archive(str(archive_base), "zip", RUN_DIR)
print("TẢI FILE:", archive)
display(FileLink(archive))
print("Lưu phiên bản riêng của notebook 08, không ghi đè notebook 07.")